# Islamic Geometric Patterns through Symmetry-Structured Neural Completion
by H. Ugail

Islamic geometric patterns are governed by exact rotational symmetry and precise constructional rules, so a completion that merely looks plausible may still be structurally invalid if its symmetry is broken. Generic neural generators treat symmetry as a soft visual property and cannot certify it. **This is an open-source reference implementation of a symmetry-structured neural completion system that takes sparse control geometry, expands it into a candidate lattice organised into rotational orbits by an exact geometric partition, and predicts edge selections and bounded curve refinements that are exactly $N$-fold symmetric by construction.** Symmetry is enforced either by an orbit-tied architecture or by inference-time orbit projection, and the orbit-tied variant carries a construction guarantee of exact $C_N$ symmetry, anchor preservation, and bounded refinement for any input and any orbit-level selection policy. The guarantee is verified by a label-free rotation audit that is independent of the identifiers the system uses internally. The repository contains the trained model checkpoints and every precomputed table, so the complete evaluation reproduces without any training.



This notebook trains and evaluates the complete symmetry-structured neural completion system reported in the paper, and regenerates all paper figures and CSV tables.

## What this notebook does

1. Sets up the runtime and the results directory (local or Google Drive).
2. Builds the procedural pattern generator, the candidate lattice, and the exact geometric orbit partition with its label-free rotation audit.
3. Trains or reuses the orbit-tied and unstructured predictors, including the corrupted-input augmented variants.
4. Calibrates selection policies and evaluates fidelity, calibration, and structural validity.
5. Runs the robustness study and the corrupted-input training control.
6. Computes the paired per-pattern statistical analysis with bootstrap confidence intervals and Wilcoxon signed-rank tests.
7. Runs the per-N generalisation and data-efficiency ablations.
8. Generates publication-quality figures, ornament renderings, SVG exports, and paper-ready CSV tables.

## Reproducibility notes

The notebook is **incremental**. Model checkpoints found in `Results/models` are loaded and their training is skipped, so a completed run is extended rather than repeated. Delete a checkpoint file to retrain that model from scratch. The cached data-efficiency table at `Results/metrics/data_efficiency.csv` is reused in the same way. For a quick functional check before a full run, set `smoke_mode = True` in the configuration cell.

## 0. Runtime and repository setup

This notebook is intended to be run either:

- locally from the root of the cloned repository; or
- in Google Colab (a GPU runtime is recommended for training).

For Colab, the recommended workflow is:

```bash
git clone https://github.com/ugail/Neural-Completion-of-Islamic-Geometric-Patterns.git
cd Symmetry-Structured-IGP-Completion
```

Then open this notebook from the repository folder. By default all outputs are written to `Results/` next to the notebook. To reuse an existing run stored on Google Drive, set `MOUNT_DRIVE = True` in the next cell and adjust `DRIVE_ROOT`.

In [ ]:
# ============================================================
# 0. Runtime and storage setup
# ============================================================
from pathlib import Path
import os, sys, platform, json, math, random, shutil, csv, time, copy
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any, Optional
import numpy as np

def in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

IN_COLAB = in_colab()
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Running in Colab:', IN_COLAB)

# Optional: keep results on Google Drive when running in Colab.
# Set MOUNT_DRIVE = True and adjust DRIVE_ROOT to extend an existing run.
MOUNT_DRIVE = False
DRIVE_ROOT = '/content/drive/.../Islamic Geometry/Code'

CODE_ROOT = None
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CODE_ROOT = Path(DRIVE_ROOT)
    CODE_ROOT.mkdir(parents=True, exist_ok=True)

if CODE_ROOT is None:
    # Default location is the repository / working directory.
    CODE_ROOT = Path(os.environ.get('IGP_CODE_ROOT', '.')).resolve()

DATASET_ROOT   = CODE_ROOT / 'wikimedia_igp_dataset'
RESULTS_ROOT   = CODE_ROOT / 'Results'
FIG_DIR        = RESULTS_ROOT / 'figures'
METRIC_DIR     = RESULTS_ROOT / 'metrics'
SVG_DIR        = RESULTS_ROOT / 'svg'
MODEL_DIR      = RESULTS_ROOT / 'models'
EXTERNAL_OUT_DIR = RESULTS_ROOT / 'external_sanity'
for d in [RESULTS_ROOT, FIG_DIR, METRIC_DIR, SVG_DIR, MODEL_DIR, EXTERNAL_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('CODE_ROOT   :', CODE_ROOT)
print('RESULTS_ROOT:', RESULTS_ROOT)

## Imports and device

In [ ]:
# ============================================================
# 1. Imports, reproducibility, and publication figure style
# ============================================================
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from PIL import Image, ImageOps, ImageDraw
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__, 'Device:', DEVICE)

def set_seed(seed: int = 0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True); return path

def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2)

def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

# ---- Consistent, professional figure style (used by every figure) ----
PALETTE = {
    'constrained':   '#1b6f8c',   # teal-blue
    'unconstrained': '#d1622b',   # warm orange
    'unc_free':      '#b23a48',   # red
    'template':      '#6f6f6f',   # grey
    'nn':            '#5c4b8a',   # purple
    'accent':        '#1d1d1d',
    'paper_bg':      '#fbfaf6',
}
def setup_plot_style():
    plt.rcParams.update({
        'figure.dpi': 120, 'savefig.dpi': 320, 'savefig.bbox': 'tight',
        'savefig.facecolor': 'white', 'figure.facecolor': 'white',
        'font.family': 'DejaVu Sans', 'font.size': 11,
        'axes.titlesize': 12.5, 'axes.titleweight': 'semibold',
        'axes.labelsize': 11.5, 'legend.fontsize': 9.5, 'legend.frameon': False,
        'xtick.labelsize': 10, 'ytick.labelsize': 10,
        'axes.spines.top': False, 'axes.spines.right': False,
        'axes.linewidth': 0.9, 'axes.grid': True, 'grid.alpha': 0.22,
        'grid.linewidth': 0.6, 'axes.axisbelow': True,
        'mathtext.default': 'regular', 'figure.constrained_layout.use': True,
    })
setup_plot_style()

def savefig_png(fig, path: Path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() != '.png':
        path = path.with_suffix('.png')
    fig.savefig(path, dpi=320, bbox_inches='tight', facecolor='white')
    print('Saved figure:', path)
    return path


## Configuration

In [ ]:
# ============================================================
# 2. Experiment configuration
# ============================================================
@dataclass
class Config:
    # ---- Data ----
    smoke_mode: bool = False     # True = fast code check only (NOT paper results)
    seed: int = 7
    train_n: int = 600
    val_n: int = 120
    test_n: int = 120
    smoke_train_n: int = 28
    smoke_val_n: int = 8
    smoke_test_n: int = 8

    # ---- Training (stabilized) ----
    epochs: int = 70
    smoke_epochs: int = 4
    lr: float = 7e-4              # lowered from 2e-3 to remove the mid-training divergence
    lr_min_ratio: float = 0.04   # cosine floor as a fraction of base lr
    warmup_epochs: int = 5       # linear LR warmup stabilizes the first steps
    grad_clip: float = 1.0
    weight_decay: float = 1e-5
    restore_best: bool = True    # restore the lowest-val-loss checkpoint at the end
    hidden_dim: int = 96
    smoke_hidden_dim: int = 32
    layers: int = 3

    # ---- Candidate lattice / domain ----
    symmetry_orders: Tuple[int, ...] = (6, 8, 10, 12)
    train_symmetry_orders: Tuple[int, ...] = (6, 8, 12)
    heldout_symmetry_order: int = 10
    num_regimes: int = 4
    num_families: int = 5
    heldout_family: int = 4
    heldout_regime: int = 3
    alpha_max: float = 0.16

    # ---- Losses ----
    bce_weight: float = 1.0
    density_weight: float = 0.35
    count_weight: float = 0.20
    alpha_straight_weight: float = 0.02
    alpha_l2_weight: float = 0.002
    pos_weight_max: float = 3.0

    # ---- Selection / calibration ----
    default_threshold: float = 0.5
    calibrate_selection: bool = True
    selection_mode: str = 'topk_ratio'
    threshold_grid_size: int = 41
    topk_ratio_min: float = 0.14
    topk_ratio_max: float = 0.55
    topk_ratio_grid_size: int = 42
    density_penalty: float = 1.15
    free_topk_ratio_min: float = 0.08
    free_topk_ratio_max: float = 0.42
    free_topk_ratio_grid_size: int = 36
    calibration_metric: str = 'f1_density'

    # ---- Corrupted-input training control ----
    train_augmented_variants: bool = True
    augment_max_dropout: float = 0.30

    # ---- Statistical analysis of the fidelity comparison ----
    equivalence_margin_f1: float = 0.02

    # ---- Incremental checkpoint reuse ----
    reuse_checkpoints: bool = True
    reuse_cached_ablations: bool = True

    # ---- Multi-seed protocol ----
    run_multi_seed: bool = True
    seeds: Tuple[int, ...] = (7, 11, 19)

    # ---- Per-N (out-of-distribution symmetry) evaluation ----
    per_n_test_n: int = 90        # test graphs generated per symmetry order

    # ---- Data-efficiency ablation ----
    run_data_efficiency: bool = True
    data_eff_sizes: Tuple[int, ...] = (100, 200, 400, 600)
    data_eff_epochs: int = 55     # slightly shorter to bound ablation runtime
    data_eff_seeds: Tuple[int, ...] = (7, 11)   # subset of seeds to bound runtime

    # ---- Imperfect-control-geometry robustness experiment ----
    # Corrupt the control geometry the network *reads* (asymmetric coordinate jitter) and
    # ask whether orbit-tied training still beats simply projecting an ordinary
    # model's output onto the orbit structure at inference. This is the decisive
    # test of whether the architectural constraint is necessary.
    run_robustness: bool = True
    robustness_modes: Tuple[str, ...] = ('jitter', 'dropout', 'sector')
    robustness_jitter: Tuple[float, ...] = (0.0, 0.02, 0.05, 0.10, 0.15)
    robustness_dropout: Tuple[float, ...] = (0.0, 0.10, 0.20, 0.30)
    robustness_sector: Tuple[float, ...] = (0.0, 0.02, 0.05, 0.10, 0.15)
    robustness_eval_n: int = 60          # test graphs used per (mode, severity)
    robustness_sym_points: int = 240     # points sampled for the symmetry metric

    # ---- Qualitative showcase ----
    gallery_best_of_n: int = 8
    gallery_linewidth: float = 1.45
    gallery_axis_limit: float = 1.38
    gallery_target_density_min: float = 0.20
    gallery_target_density_max: float = 0.62
    save_gallery_svgs: bool = True

CFG = Config()
if CFG.smoke_mode:
    CFG.train_n, CFG.val_n, CFG.test_n = CFG.smoke_train_n, CFG.smoke_val_n, CFG.smoke_test_n
    CFG.epochs = CFG.smoke_epochs
    CFG.warmup_epochs = 1
    CFG.hidden_dim = CFG.smoke_hidden_dim
    CFG.layers = 1
    CFG.seeds = (7,)
    CFG.per_n_test_n = 8
    CFG.data_eff_sizes = (24, 48)
    CFG.data_eff_epochs = 3
    CFG.data_eff_seeds = (7,)
    CFG.robustness_jitter = (0.0, 0.1)
    CFG.robustness_dropout = (0.0, 0.2)
    CFG.robustness_sector = (0.0, 0.1)
    CFG.robustness_eval_n = 4
    CFG.robustness_sym_points = 120

set_seed(CFG.seed)
print('PAPER MODE' if not CFG.smoke_mode else 'SMOKE MODE (code check only)')
print(CFG)


## Geometric primitives

In [ ]:
# ============================================================
# 3. Geometry helpers
# ============================================================

TWO_PI = 2 * math.pi

def rotmat(theta: float) -> np.ndarray:
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s], [s, c]], dtype=np.float32)

def polar(r: float, theta: float) -> np.ndarray:
    return np.array([r * math.cos(theta), r * math.sin(theta)], dtype=np.float32)

def angle_of(p: np.ndarray) -> float:
    return math.atan2(float(p[1]), float(p[0]))

def radius_of(p: np.ndarray) -> float:
    return float(np.linalg.norm(p))

def angle_diff(a: float, b: float) -> float:
    d = (a - b + math.pi) % (2 * math.pi) - math.pi
    return d

def normalize_points(points: np.ndarray) -> np.ndarray:
    pts = points.copy()
    mn = pts.min(axis=0)
    mx = pts.max(axis=0)
    scale = max(float((mx - mn).max()), 1e-8)
    pts = (pts - (mn + mx) / 2.0) / scale
    return pts.astype(np.float32)

def unique_edges(edges: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    s = set()
    out = []
    for a, b in edges:
        if a == b:
            continue
        e = (a, b) if a < b else (b, a)
        if e not in s:
            s.add(e)
            out.append(e)
    return out

def line_segments_from_edges(points: np.ndarray, edges: List[Tuple[int, int]]) -> np.ndarray:
    if len(edges) == 0:
        return np.zeros((0, 2, 2), dtype=np.float32)
    return np.array([[points[i], points[j]] for i, j in edges], dtype=np.float32)

## Data structures

In [ ]:
# ============================================================
# 4. Scaffold, candidate lattice and target graph data classes
# ============================================================

EDGE_TYPES = {
    'ring': 0,
    'radial': 1,
    'skip': 2,
    'diag': 3,
    'boundary': 4,
    'star_cross': 5,
    'secondary': 6,
    'border': 7,
}
EDGE_TYPE_NAMES = {v: k for k, v in EDGE_TYPES.items()}
NUM_EDGE_TYPES = len(EDGE_TYPES)

@dataclass
class PatternSample:
    sample_id: str
    scaffold_type: str
    N: int
    family_id: int
    regime_id: int
    points: np.ndarray                  # [V, 2]
    node_features: np.ndarray           # [V, Fv]
    edges: np.ndarray                   # [E, 2]
    edge_features: np.ndarray           # [E, Fe]
    edge_types: np.ndarray              # [E]
    edge_labels: np.ndarray             # [E], 0/1 target
    orbit_ids: np.ndarray               # [E]
    anchor_mask: np.ndarray             # [V]
    target_edges: List[Tuple[int, int]]
    scaffold_info: Dict[str, Any]

## Control geometry and candidate lattice

In [ ]:
# ============================================================
# 5. Unified scaffold-implied candidate lattice
# ============================================================

def make_scaffold(
    N: int,
    scaffold_type: str,
    rng: np.random.Generator,
    outer_radius: float = 1.0,
) -> Dict[str, Any]:
    """Create minimal or anchored scaffold metadata.

    Minimal scaffold:
        center + symmetry order + boundary radius.
    Anchored scaffold:
        center + outer star tips + inner valley anchors + boundary.
    """
    assert scaffold_type in ['minimal', 'anchored']
    inner_ratio = float(rng.uniform(0.36, 0.58))
    mid_ratio = float(rng.uniform(0.62, 0.78))
    small_ratio = float(rng.uniform(0.20, 0.32))
    phase = float(rng.choice([0.0, math.pi / N]))

    info = {
        'N': int(N),
        'scaffold_type': scaffold_type,
        'outer_radius': float(outer_radius),
        'inner_ratio': inner_ratio,
        'mid_ratio': mid_ratio,
        'small_ratio': small_ratio,
        'phase': phase,
        'center': [0.0, 0.0],
    }
    return info

def build_candidate_lattice(scaffold: Dict[str, Any]) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, List[Dict[str, Any]]]:
    """Build a unified candidate lattice closed under C_N.

    The lattice is deliberately a superset across target families. Families are represented by
    different target masks over this same candidate space, not by separate candidate generators.
    """
    N = int(scaffold['N'])
    scaffold_type = scaffold['scaffold_type']
    outer = float(scaffold['outer_radius'])
    inner = float(scaffold['inner_ratio']) * outer
    mid = float(scaffold['mid_ratio']) * outer
    small = float(scaffold['small_ratio']) * outer

    # ring specifications: (name, radius, phase_kind)
    # phase_kind 0: sector angles i/N, phase_kind 1: half-sector angles (i+0.5)/N
    ring_specs = [
        ('small_0', small, 0),
        ('small_h', small, 1),
        ('inner_0', inner, 0),
        ('inner_h', inner, 1),
        ('mid_0', mid, 0),
        ('mid_h', mid, 1),
        ('outer_0', outer, 0),
        ('outer_h', outer, 1),
    ]

    points = []
    labels = []
    anchor_mask = []

    # center
    points.append(np.array([0.0, 0.0], dtype=np.float32))
    labels.append({'kind': 'center', 'ring': -1, 'phase': 0, 'sector': -1, 'r': 0.0, 'theta': 0.0})
    anchor_mask.append(True)

    idx_by_key = {('center', -1, 0, -1): 0}

    for ring_idx, (ring_name, r, phase_kind) in enumerate(ring_specs):
        for i in range(N):
            theta = TWO_PI * (i + 0.5 * phase_kind) / N
            idx = len(points)
            points.append(polar(r, theta))
            labels.append({'kind': ring_name, 'ring': ring_idx, 'phase': phase_kind, 'sector': i, 'r': r, 'theta': theta})
            # Anchored scaffolds have outer star tips and inner valley anchors fixed.
            is_anchor = False
            if scaffold_type == 'anchored':
                if ring_name in ['outer_0', 'inner_h']:
                    is_anchor = True
            else:
                # Minimal scaffolds only fix the boundary ring and center.
                if ring_name == 'outer_0':
                    is_anchor = True
            anchor_mask.append(is_anchor)
            idx_by_key[(ring_name, ring_idx, phase_kind, i)] = idx

    points = np.array(points, dtype=np.float32)
    anchor_mask = np.array(anchor_mask, dtype=bool)

    # Helpers
    def ring_indices_by_name(name):
        return [i for i, lab in enumerate(labels) if lab['kind'] == name]

    edges = []
    edge_types = []

    def add_edge(a, b, typ):
        if a == b:
            return
        edges.append((a, b) if a < b else (b, a))
        edge_types.append(EDGE_TYPES[typ])

    # Ring edges
    for ring_name, _, _ in ring_specs:
        inds = ring_indices_by_name(ring_name)
        inds_by_sector = {labels[idx]['sector']: idx for idx in inds}
        for i in range(N):
            add_edge(inds_by_sector[i], inds_by_sector[(i + 1) % N], 'ring')

    # Boundary edges more strongly typed
    for ring_name in ['outer_0', 'outer_h']:
        inds = ring_indices_by_name(ring_name)
        bysec = {labels[idx]['sector']: idx for idx in inds}
        for i in range(N):
            add_edge(bysec[i], bysec[(i + 1) % N], 'boundary')

    # Radial edges center-to-rings and ring-to-ring same sector/phase
    center_idx = 0
    radial_names = ['small_0', 'inner_0', 'mid_0', 'outer_0']
    radial_h_names = ['small_h', 'inner_h', 'mid_h', 'outer_h']
    for names in [radial_names, radial_h_names]:
        prev = None
        for name in names:
            bysec = {labels[idx]['sector']: idx for idx in ring_indices_by_name(name)}
            for i in range(N):
                if prev is None:
                    add_edge(center_idx, bysec[i], 'radial')
                else:
                    add_edge(prev[i], bysec[i], 'radial')
            prev = bysec

    # Skip-k star links on selected rings. k values 2 and 3 where meaningful.
    for ring_name in ['inner_0', 'inner_h', 'mid_0', 'outer_0']:
        bysec = {labels[idx]['sector']: idx for idx in ring_indices_by_name(ring_name)}
        for k in [2, 3]:
            if k < N // 2 + 1:
                for i in range(N):
                    add_edge(bysec[i], bysec[(i + k) % N], 'skip')

    # Cross-ring diagonals
    pairs = [
        ('small_0', 'inner_h'), ('small_h', 'inner_0'),
        ('inner_0', 'mid_h'), ('inner_h', 'mid_0'),
        ('mid_0', 'outer_h'), ('mid_h', 'outer_0'),
        ('inner_h', 'outer_0'), ('inner_0', 'outer_h')
    ]
    for a_name, b_name in pairs:
        A = {labels[idx]['sector']: idx for idx in ring_indices_by_name(a_name)}
        B = {labels[idx]['sector']: idx for idx in ring_indices_by_name(b_name)}
        for i in range(N):
            add_edge(A[i], B[i], 'diag')
            add_edge(A[i], B[(i + 1) % N], 'diag')

    # Star-cross connectors
    A = {labels[idx]['sector']: idx for idx in ring_indices_by_name('inner_h')}
    B = {labels[idx]['sector']: idx for idx in ring_indices_by_name('outer_0')}
    C = {labels[idx]['sector']: idx for idx in ring_indices_by_name('mid_h')}
    for i in range(N):
        add_edge(A[i], B[i], 'star_cross')
        add_edge(A[i], B[(i + 1) % N], 'star_cross')
        add_edge(C[i], B[i], 'star_cross')
        add_edge(C[i], B[(i + 1) % N], 'star_cross')

    # Secondary connectors
    for a_name, b_name in [('small_0', 'mid_0'), ('small_h', 'mid_h'), ('small_0', 'outer_0')]:
        A = {labels[idx]['sector']: idx for idx in ring_indices_by_name(a_name)}
        B = {labels[idx]['sector']: idx for idx in ring_indices_by_name(b_name)}
        for i in range(N):
            add_edge(A[i], B[(i + 1) % N], 'secondary')
            add_edge(A[i], B[(i - 1) % N], 'secondary')

    # Border connectors
    A = {labels[idx]['sector']: idx for idx in ring_indices_by_name('mid_0')}
    B = {labels[idx]['sector']: idx for idx in ring_indices_by_name('outer_h')}
    for i in range(N):
        add_edge(A[i], B[i], 'border')
        add_edge(A[i], B[(i + 1) % N], 'border')

    # Unique edges while preserving first edge type
    type_by_edge = {}
    for e, t in zip(edges, edge_types):
        if e not in type_by_edge:
            type_by_edge[e] = t
    edges_unique = list(type_by_edge.keys())
    edge_types_unique = np.array([type_by_edge[e] for e in edges_unique], dtype=np.int64)
    edges_np = np.array(edges_unique, dtype=np.int64)

    # Node features
    node_features = []
    for i, p in enumerate(points):
        r = radius_of(p)
        th = angle_of(p)
        lab = labels[i]
        node_features.append([
            p[0], p[1],
            r,
            math.cos(th), math.sin(th),
            float(anchor_mask[i]),
            (lab['ring'] + 1) / max(1, len(ring_specs)),
            float(lab['phase']),
            float(scaffold_type == 'anchored'),
            N / 12.0,
        ])
    node_features = np.array(node_features, dtype=np.float32)

    # Edge features
    edge_features = []
    for (a, b), typ in zip(edges_unique, edge_types_unique):
        pa, pb = points[a], points[b]
        ra, rb = radius_of(pa), radius_of(pb)
        ta, tb = angle_of(pa), angle_of(pb)
        d = pb - pa
        length = float(np.linalg.norm(d))
        ad = abs(angle_diff(tb, ta)) / math.pi
        rd = abs(rb - ra)
        type_oh = [0.0] * NUM_EDGE_TYPES
        type_oh[int(typ)] = 1.0
        edge_features.append([
            d[0], d[1], length, rd, ad,
            ra, rb,
            float(N) / 12.0,
            float(scaffold_type == 'anchored'),
        ] + type_oh)
    edge_features = np.array(edge_features, dtype=np.float32)

    return points, node_features, edges_np, edge_features, edge_types_unique, anchor_mask, labels

def compute_edge_orbits_from_labels(
    edges: np.ndarray,
    edge_types: np.ndarray,
    labels: List[Dict[str, Any]],
    N: int
) -> np.ndarray:
    """Compute cyclic edge orbits using lattice labels.

    Canonicalises an edge by rotating sectors so the minimum sector is 0.
    Center endpoints are handled separately.
    """
    orbit_keys = []
    for (a, b), typ in zip(edges, edge_types):
        la, lb = labels[int(a)], labels[int(b)]
        def token(lab):
            return (lab['kind'], int(lab['ring']), int(lab['phase']), int(lab['sector']))
        ta, tb = token(la), token(lb)

        sectors = [s for s in [ta[3], tb[3]] if s >= 0]
        if len(sectors) == 0:
            shift = 0
        else:
            shift = min(sectors)

        def shift_token(t):
            kind, ring, phase, sec = t
            sec2 = -1 if sec < 0 else (sec - shift) % N
            return (kind, ring, phase, sec2)

        ca, cb = shift_token(ta), shift_token(tb)
        pair = tuple(sorted([ca, cb]))
        orbit_keys.append((int(typ), pair))

    key_to_id = {}
    ids = []
    for k in orbit_keys:
        if k not in key_to_id:
            key_to_id[k] = len(key_to_id)
        ids.append(key_to_id[k])
    return np.array(ids, dtype=np.int64)

## Target patterns and samples

In [ ]:
# ============================================================
# 6. Stochastic target distribution
# ============================================================

def target_rule_prob(
    edge_type: int,
    family_id: int,
    regime_id: int,
    rng: np.random.Generator,
) -> float:
    """Template-centred stochastic target probability for an edge type.

    The target distribution must not be a purely independent Bernoulli draw over the
    whole candidate lattice. Each family/regime pair defines a coherent template over
    edge types. Stochasticity then perturbs that template at the orbit level. This
    gives controlled variation without turning the target into an arbitrary random
    subset of candidate edges.

    Family/regime names are neutral procedural regimes, not historical/cultural labels.
    """
    name = EDGE_TYPE_NAMES[int(edge_type)]

    # Coherent family templates. Values are probabilities for orbit inclusion.
    # They are deliberately sparse enough that the model must learn edge selection
    # instead of being rewarded for selecting the entire candidate lattice.
    family_templates = {
        0: {'ring': .72, 'radial': .30, 'skip': .70, 'diag': .08, 'boundary': 1.00, 'star_cross': .06, 'secondary': .12, 'border': .18},
        1: {'ring': .34, 'radial': .16, 'skip': .28, 'diag': .36, 'boundary': 1.00, 'star_cross': .78, 'secondary': .14, 'border': .20},
        2: {'ring': .55, 'radial': .22, 'skip': .38, 'diag': .10, 'boundary': 1.00, 'star_cross': .15, 'secondary': .20, 'border': .72},
        3: {'ring': .30, 'radial': .12, 'skip': .62, 'diag': .58, 'boundary': 1.00, 'star_cross': .34, 'secondary': .44, 'border': .22},
        4: {'ring': .76, 'radial': .34, 'skip': .32, 'diag': .18, 'boundary': 1.00, 'star_cross': .22, 'secondary': .58, 'border': .44},
    }

    # Regime modifications. These create controlled multi-regime synthesis from the
    # same scaffold without making unsupported historical-style claims.
    regime_mods = {
        # A: ring-dominant / calm rosette
        0: {'ring': +.18, 'radial': +.05, 'skip': -.12, 'diag': -.16, 'star_cross': -.18, 'secondary': -.10, 'border': -.06},
        # B: sharp star / radial connector emphasis
        1: {'ring': -.02, 'radial': +.18, 'skip': +.18, 'diag': -.02, 'star_cross': -.04, 'secondary': -.04, 'border': -.08},
        # C: star-cross / diagonal emphasis
        2: {'ring': -.16, 'radial': -.06, 'skip': -.08, 'diag': +.24, 'star_cross': +.32, 'secondary': -.02, 'border': -.10},
        # D: nested/border/secondary emphasis
        3: {'ring': +.04, 'radial': -.02, 'skip': +.00, 'diag': +.06, 'star_cross': +.02, 'secondary': +.26, 'border': +.26},
    }

    p = family_templates[int(family_id)].get(name, 0.05)
    p += regime_mods[int(regime_id)].get(name, 0.0)

    # Light orbit-level randomness. Keep this small: coherence comes from templates,
    # not from unstructured random edge selection.
    p += float(rng.normal(0.0, 0.020))
    return float(np.clip(p, 0.01, 1.00))


def _orbits_for_type(edge_types: np.ndarray, orbit_ids: np.ndarray, type_name: str) -> np.ndarray:
    typ = EDGE_TYPES[type_name]
    idx = np.where(edge_types == typ)[0]
    if len(idx) == 0:
        return np.array([], dtype=int)
    return np.unique(orbit_ids[idx]).astype(int)


def _activate_orbits(y: np.ndarray, orbit_ids: np.ndarray, chosen_orbits: List[int]):
    chosen = set(int(o) for o in chosen_orbits)
    if not chosen:
        return
    mask = np.array([int(o) in chosen for o in orbit_ids], dtype=bool)
    y[mask] = 1.0


def generate_target_labels(
    edge_types: np.ndarray,
    orbit_ids: np.ndarray,
    family_id: int,
    regime_id: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """Generate orbit-consistent, template-centred stochastic target edge labels.

    Important design choice:
    - Every stochastic decision is made at the orbit level.
    - The fallback for sparse graphs activates whole orbits, not individual edges.
    This preserves the cyclic target symmetry needed for fair training of the
    constrained model.
    """
    y = np.zeros(len(edge_types), dtype=np.float32)
    unique_orbits = np.unique(orbit_ids).astype(int)

    # Global optional gates create coherent sample-level variation. If a gate is off,
    # the corresponding edge family is strongly suppressed throughout the graph.
    gates = {
        'diag': rng.random() < (0.24 + 0.14 * (int(family_id) in [1, 3]) + 0.36 * (int(regime_id) == 2) - 0.10 * (int(regime_id) == 0)),
        'star_cross': rng.random() < (0.18 + 0.24 * (int(family_id) == 1) + 0.42 * (int(regime_id) == 2) - 0.08 * (int(regime_id) == 0)),
        'secondary': rng.random() < (0.22 + 0.16 * (int(family_id) in [3, 4]) + 0.38 * (int(regime_id) == 3)),
        'border': rng.random() < (0.34 + 0.22 * (int(family_id) == 2) + 0.42 * (int(regime_id) == 3)),
    }

    for oid in unique_orbits:
        idxs = np.where(orbit_ids == oid)[0]
        typ = int(edge_types[idxs[0]])
        name = EDGE_TYPE_NAMES[typ]
        p = target_rule_prob(typ, family_id, regime_id, rng)

        # Coherent optional structures: suppress optional types when their sample-level
        # gate is off, rather than independently flickering many orbits on/off.
        if name in gates and not gates[name]:
            p *= 0.18

        include = rng.random() < p
        y[idxs] = 1.0 if include else 0.0

    # Boundary orbits are always active.
    boundary_orbits = _orbits_for_type(edge_types, orbit_ids, 'boundary')
    _activate_orbits(y, orbit_ids, list(boundary_orbits))

    # Ensure coherent internal structure, using whole orbits only.
    min_edges = max(8, int(round(0.10 * len(y))))
    if int(y.sum()) < min_edges:
        fallback_types = ['ring', 'skip', 'radial']
        for typ_name in fallback_types:
            oids = _orbits_for_type(edge_types, orbit_ids, typ_name)
            if len(oids) == 0:
                continue
            k = max(1, int(math.ceil(0.18 * len(oids))))
            chosen = rng.choice(oids, size=min(k, len(oids)), replace=False)
            _activate_orbits(y, orbit_ids, list(map(int, chosen)))
            if int(y.sum()) >= min_edges:
                break

    # Defensive invariant: all labels are constant within each orbit.
    # This is cheap and catches accidental target asymmetry immediately.
    for oid in unique_orbits:
        idxs = np.where(orbit_ids == oid)[0]
        if len(idxs) > 1:
            # If numerical or programming errors ever create mixed labels, use majority.
            val = 1.0 if y[idxs].mean() >= 0.5 else 0.0
            y[idxs] = val

    return y.astype(np.float32)

def make_sample(
    sample_id: str,
    N: int,
    scaffold_type: str,
    family_id: int,
    regime_id: int,
    rng: np.random.Generator,
) -> PatternSample:
    scaffold = make_scaffold(N, scaffold_type, rng)
    points, node_features, edges, edge_features, edge_types, anchor_mask, labels = build_candidate_lattice(scaffold)
    orbit_ids = compute_edge_orbits_from_labels(edges, edge_types, labels, N)
    y = generate_target_labels(edge_types, orbit_ids, family_id, regime_id, rng)
    target_edges = [tuple(map(int, e)) for e, yy in zip(edges, y) if yy > 0.5]

    info = dict(scaffold)
    info.update({'labels': labels})
    return PatternSample(
        sample_id=sample_id,
        scaffold_type=scaffold_type,
        N=N,
        family_id=family_id,
        regime_id=regime_id,
        points=points,
        node_features=node_features,
        edges=edges,
        edge_features=edge_features,
        edge_types=edge_types,
        edge_labels=y,
        orbit_ids=orbit_ids,
        anchor_mask=anchor_mask,
        target_edges=target_edges,
        scaffold_info=info,
    )

def make_dataset(
    n: int,
    seed: int,
    split: str,
    scaffold_types=('anchored', 'minimal'),
    symmetry_orders=(6, 8, 10, 12),
    families=(0,1,2,3,4),
    regimes=(0,1,2,3),
) -> List[PatternSample]:
    rng = np.random.default_rng(seed)
    data = []
    for i in range(n):
        N = int(rng.choice(symmetry_orders))
        scaffold_type = str(rng.choice(scaffold_types, p=[0.72, 0.28] if len(scaffold_types)==2 else None))
        family_id = int(rng.choice(families))
        regime_id = int(rng.choice(regimes))
        data.append(make_sample(f'{split}_{i:05d}', N, scaffold_type, family_id, regime_id, rng))
    return data

## Exact geometric orbit partition (label-free)

In [ ]:
# ============================================================
# 7b. Exact geometric orbit partition (label-free)
# ============================================================
# Candidate edges are grouped into rotational orbits by an exact geometric
# procedure rather than by symbolic labels: the C_N rotation map is computed on
# the vertex coordinates, every edge is connected to its rotation image, and the
# connected components of that relation (union-find) are the true orbits. A
# label-free audit is also provided: it rotates every selected edge by exactly
# 2*pi/N and checks that the image edge is selected, so the verification of the
# symmetry guarantee is independent of the identifiers the system itself uses.

def rotation_vertex_map(pts, N, tol=1e-4):
    c, s = math.cos(2 * math.pi / N), math.sin(2 * math.pi / N)
    R = np.array([[c, -s], [s, c]])
    rot = np.asarray(pts, dtype=np.float64) @ R.T
    m = np.full(len(pts), -1, dtype=int)
    for i, p in enumerate(rot):
        d = np.linalg.norm(np.asarray(pts, dtype=np.float64) - p, axis=1)
        j = int(np.argmin(d))
        if d[j] < tol:
            m[i] = j
    return m

def fix_orbit_ids(sample):
    """Replace label-based orbit identifiers with the exact geometric partition."""
    vm = rotation_vertex_map(sample.points, int(sample.N))
    assert (vm >= 0).all(), 'candidate lattice is not closed under C_N rotation'
    eidx = {frozenset((int(a), int(b))): k for k, (a, b) in enumerate(sample.edges)}
    parent = list(range(len(sample.edges)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for k, (a, b) in enumerate(sample.edges):
        img = eidx[frozenset((int(vm[int(a)]), int(vm[int(b)])))]
        ra, rb = find(k), find(img)
        if ra != rb:
            parent[rb] = ra
    roots = [find(k) for k in range(len(sample.edges))]
    remap = {r: i for i, r in enumerate(dict.fromkeys(roots))}
    s2 = copy.copy(sample)
    s2.orbit_ids = np.array([remap[r] for r in roots], dtype=sample.orbit_ids.dtype)
    return s2

def true_rotation_violations(sample, selected):
    """Label-free audit: selected edges whose exact rotation image is unselected."""
    vm = rotation_vertex_map(sample.points, int(sample.N))
    eidx = {frozenset((int(a), int(b))): k for k, (a, b) in enumerate(sample.edges)}
    bad = 0
    for k, (a, b) in enumerate(sample.edges):
        if selected[k]:
            img = eidx[frozenset((int(vm[int(a)]), int(vm[int(b)])))]
            if not selected[img]:
                bad += 1
    return bad

_orig_make_sample = make_sample
def make_sample(*args, **kwargs):
    return fix_orbit_ids(_orig_make_sample(*args, **kwargs))
print('Orbit identifiers: exact geometric partition (union-find under the C_N rotation map).')


## Orbit operations

In [ ]:
# ============================================================
# 7. PyTorch conversion and orbitwise projection
# ============================================================

def sample_to_torch(sample: PatternSample, device=DEVICE) -> Dict[str, torch.Tensor]:
    return {
        'node_features': torch.tensor(sample.node_features, dtype=torch.float32, device=device),
        'edge_index': torch.tensor(sample.edges.T, dtype=torch.long, device=device),  # [2, E]
        'edge_features': torch.tensor(sample.edge_features, dtype=torch.float32, device=device),
        'edge_labels': torch.tensor(sample.edge_labels, dtype=torch.float32, device=device),
        'edge_types': torch.tensor(sample.edge_types, dtype=torch.long, device=device),
        'orbit_ids': torch.tensor(sample.orbit_ids, dtype=torch.long, device=device),
        'anchor_mask': torch.tensor(sample.anchor_mask.astype(np.float32), dtype=torch.float32, device=device),
    }

def orbit_average(values: torch.Tensor, orbit_ids: torch.Tensor) -> torch.Tensor:
    """Average values over integer orbit IDs."""
    out = torch.zeros_like(values)
    unique = torch.unique(orbit_ids)
    for oid in unique:
        mask = orbit_ids == oid
        out[mask] = values[mask].mean()
    return out

## Edge message-passing model

In [ ]:
# ============================================================
# 8. Neural model
# ============================================================

class EdgeMessagePassing(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, hidden_dim: int, layers: int = 3, num_regimes: int = 4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layers = layers
        self.node_in = nn.Sequential(
            nn.Linear(node_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.regime_emb = nn.Embedding(num_regimes, hidden_dim)

        self.msg_layers = nn.ModuleList()
        self.upd_layers = nn.ModuleList()
        for _ in range(layers):
            self.msg_layers.append(nn.Sequential(
                nn.Linear(hidden_dim * 2 + edge_dim, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, hidden_dim),
            ))
            self.upd_layers.append(nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, hidden_dim),
            ))

        self.edge_head = nn.Sequential(
            nn.Linear(hidden_dim * 3 + edge_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.alpha_head = nn.Sequential(
            nn.Linear(hidden_dim * 3 + edge_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Tanh(),
        )

    def forward(self, batch: Dict[str, torch.Tensor], regime_id: int, constrained: bool = True):
        x = batch['node_features']
        edge_index = batch['edge_index']
        edge_attr = batch['edge_features']
        orbit_ids = batch['orbit_ids']

        src, dst = edge_index[0], edge_index[1]
        h = self.node_in(x)

        for msg, upd in zip(self.msg_layers, self.upd_layers):
            # undirected message passing: send both directions
            m1 = msg(torch.cat([h[src], h[dst], edge_attr], dim=-1))
            m2 = msg(torch.cat([h[dst], h[src], edge_attr], dim=-1))
            agg = torch.zeros_like(h)
            agg.index_add_(0, dst, m1)
            agg.index_add_(0, src, m2)
            deg = torch.zeros((h.shape[0], 1), device=h.device)
            one = torch.ones((src.shape[0], 1), device=h.device)
            deg.index_add_(0, dst, one)
            deg.index_add_(0, src, one)
            agg = agg / deg.clamp(min=1.0)
            h = h + upd(torch.cat([h, agg], dim=-1))

        z = self.regime_emb(torch.tensor([regime_id], dtype=torch.long, device=x.device)).expand(edge_attr.shape[0], -1)
        edge_context = torch.cat([h[src], h[dst], z, edge_attr], dim=-1)

        logits = self.edge_head(edge_context).squeeze(-1)
        alpha_raw = self.alpha_head(edge_context).squeeze(-1)

        if constrained:
            logits = orbit_average(logits, orbit_ids)
            alpha_raw = orbit_average(alpha_raw, orbit_ids)

        return logits, alpha_raw

## Loss

In [ ]:
# ============================================================
# 9. Loss functions
# ============================================================

def model_loss(
    logits: torch.Tensor,
    alpha_raw: torch.Tensor,
    batch: Dict[str, torch.Tensor],
    cfg: Config,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    y = batch['edge_labels']
    edge_types = batch['edge_types']

    # Class imbalance: compute per-sample pos_weight.
    pos = y.sum().clamp(min=1.0)
    neg = (1.0 - y).sum().clamp(min=1.0)
    # Too-large positive weights can push early training into the degenerate
    # "predict everything" solution. Clamp conservatively and add an explicit
    # count-matching penalty below.
    pos_weight = (neg / pos).detach().clamp(0.5, cfg.pos_weight_max)
    bce = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)

    p = torch.sigmoid(logits)
    density = torch.abs(p.sum() - y.sum()) / y.numel()
    count_match = ((p.sum() - y.sum()) / y.numel()).pow(2)

    structural_mask = (
        (edge_types == EDGE_TYPES['ring']) |
        (edge_types == EDGE_TYPES['radial']) |
        (edge_types == EDGE_TYPES['boundary'])
    ).float()
    alpha_straight = (structural_mask * alpha_raw.pow(2)).mean()
    alpha_l2 = alpha_raw.pow(2).mean()

    loss = (
        cfg.bce_weight * bce
        + cfg.density_weight * density
        + cfg.count_weight * count_match
        + cfg.alpha_straight_weight * alpha_straight
        + cfg.alpha_l2_weight * alpha_l2
    )

    return loss, {
        'bce': float(bce.detach().cpu()),
        'density': float(density.detach().cpu()),
        'count_match': float(count_match.detach().cpu()),
        'pos_weight': float(pos_weight.detach().cpu()),
        'alpha_straight': float(alpha_straight.detach().cpu()),
        'alpha_l2': float(alpha_l2.detach().cpu()),
    }

## Metrics

In [ ]:
# ============================================================
# 10. Refinement, prediction and metrics
# ============================================================

def select_edges_from_scores(
    scores: np.ndarray,
    orbit_ids: np.ndarray,
    threshold: float = 0.5,
    topk_orbits: Optional[int] = None,
    topk_ratio: Optional[float] = None,
) -> np.ndarray:
    """Select candidate edges from scores.

    Selection is orbitwise whenever topk_orbits or topk_ratio is used. This is
    the preferred policy for the constrained synthesis experiments because it
    controls edge density and avoids the low-threshold, select-everything failure
    mode observed in early smoke runs.
    """
    orbit_ids = np.asarray(orbit_ids)
    scores = np.asarray(scores, dtype=np.float32)

    if topk_ratio is not None:
        unique = np.unique(orbit_ids)
        topk_orbits = max(1, int(round(float(topk_ratio) * len(unique))))

    if topk_orbits is not None:
        unique = np.unique(orbit_ids)
        orbit_scores = []
        for oid in unique:
            orbit_scores.append((oid, float(scores[orbit_ids == oid].mean())))
        orbit_scores.sort(key=lambda x: x[1], reverse=True)
        k = max(1, min(int(topk_orbits), len(orbit_scores)))
        chosen = set([oid for oid, _ in orbit_scores[:k]])
        return np.array([oid in chosen for oid in orbit_ids], dtype=bool)

    return scores >= threshold

def policy_to_label(policy: Dict[str, float]) -> str:
    mode = policy.get('mode')
    if mode == 'topk_ratio':
        return f"orbit topK={policy.get('topk_ratio', 0):.2f}"
    if mode == 'free_topk_ratio':
        return f"free topK={policy.get('topk_ratio', 0):.2f}"
    if mode == 'free_threshold':
        return f"free τ={policy.get('threshold', 0.5):.2f}"
    return f"τ={policy.get('threshold', 0.5):.2f}"

def select_with_policy(scores: np.ndarray, orbit_ids: np.ndarray, policy: Dict[str, float]) -> np.ndarray:
    """Select candidate edges under a named policy.

    Modes:
      threshold       : independent thresholding on edge scores.
      topk_ratio      : orbitwise top-K selection, preserving cyclic edge orbits.
      free_threshold  : unconstrained independent thresholding; used to expose
                        symmetry violations in the unconstrained baseline.
      free_topk_ratio : unconstrained individual-edge top-K selection; controls
                        density but does not force orbit closure.
    """
    mode = policy.get('mode', 'threshold')
    scores = np.asarray(scores, dtype=np.float32)
    if mode == 'topk_ratio':
        return select_edges_from_scores(scores, orbit_ids, topk_ratio=policy.get('topk_ratio', 0.35))
    if mode == 'free_threshold':
        return scores >= float(policy.get('threshold', 0.5))
    if mode == 'free_topk_ratio':
        ratio = float(policy.get('topk_ratio', 0.25))
        k = max(1, min(len(scores), int(round(ratio * len(scores)))))
        idx = np.argsort(-scores)[:k]
        out = np.zeros(len(scores), dtype=bool)
        out[idx] = True
        return out
    return select_edges_from_scores(scores, orbit_ids, threshold=policy.get('threshold', 0.5))

def calibration_objective(metrics: Dict[str, float], cfg: Config) -> float:
    """Validation objective for threshold/top-K calibration.

    Raw F1 can reward very low thresholds that select almost all edges. The
    density-aware objective penalises edge-density error, improving precision and
    figure quality without using target edges at inference time.
    """
    if cfg.calibration_metric == 'f1':
        return float(metrics['f1'])
    if cfg.calibration_metric == 'balanced':
        return float(0.5 * metrics['f1'] + 0.5 * metrics['precision_at_k'] - cfg.density_penalty * metrics['edge_density_error'])
    # default: density-aware F1
    return float(metrics['f1'] - cfg.density_penalty * metrics['edge_density_error'])

def refined_edge_polylines(
    points: np.ndarray,
    edges: np.ndarray,
    selected: np.ndarray,
    alpha: np.ndarray,
    edge_types: np.ndarray,
    alpha_max: float,
    samples_per_edge: int = 8,
) -> List[np.ndarray]:
    """Return polylines for selected edges. Structural edges remain straight."""
    polylines = []
    for idx, sel in enumerate(selected):
        if not sel:
            continue
        a, b = map(int, edges[idx])
        p0, p1 = points[a], points[b]
        typ = int(edge_types[idx])
        if typ in [EDGE_TYPES['ring'], EDGE_TYPES['radial'], EDGE_TYPES['boundary']]:
            polylines.append(np.stack([p0, p1], axis=0))
            continue
        v = p1 - p0
        L = np.linalg.norm(v) + 1e-9
        n = np.array([-v[1], v[0]], dtype=np.float32) / L
        m = 0.5 * (p0 + p1)
        aa = float(np.tanh(alpha[idx]) * alpha_max)
        q = m + aa * L * n
        ts = np.linspace(0, 1, samples_per_edge)
        curve = []
        for t in ts:
            # Quadratic Bezier through q.
            pt = (1-t)**2 * p0 + 2*(1-t)*t * q + t**2 * p1
            curve.append(pt)
        polylines.append(np.array(curve, dtype=np.float32))
    return polylines

def sample_points_from_polylines(polylines: List[np.ndarray], max_points: int = 500) -> np.ndarray:
    pts = []
    for poly in polylines:
        if len(poly) == 0:
            continue
        pts.extend(poly.tolist())
    if len(pts) == 0:
        return np.zeros((1, 2), dtype=np.float32)
    pts = np.array(pts, dtype=np.float32)
    if len(pts) > max_points:
        idx = np.linspace(0, len(pts)-1, max_points).astype(int)
        pts = pts[idx]
    return pts

def sample_points_from_edges(points: np.ndarray, edges: List[Tuple[int, int]], samples_per_edge: int = 6, max_points: int = 600) -> np.ndarray:
    pts = []
    for a, b in edges:
        p0, p1 = points[int(a)], points[int(b)]
        for t in np.linspace(0, 1, samples_per_edge):
            pts.append((1-t)*p0 + t*p1)
    if not pts:
        return np.zeros((1,2), dtype=np.float32)
    pts = np.array(pts, dtype=np.float32)
    if len(pts) > max_points:
        idx = np.linspace(0, len(pts)-1, max_points).astype(int)
        pts = pts[idx]
    return pts

def chamfer_distance_np(A: np.ndarray, B: np.ndarray) -> float:
    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)
    if len(A) == 0 or len(B) == 0:
        return float('inf')
    # Pairwise distances, small point sets.
    D = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1)
    return float(np.sqrt(D.min(axis=1)).mean() + np.sqrt(D.min(axis=0)).mean()) / 2.0

def hausdorff_np(A: np.ndarray, B: np.ndarray) -> float:
    D = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=-1)
    return float(max(np.sqrt(D.min(axis=1)).max(), np.sqrt(D.min(axis=0)).max()))

def rotational_self_chamfer(points: np.ndarray, N: int) -> float:
    vals = []
    for k in range(1, N):
        R = rotmat(TWO_PI * k / N)
        P2 = points @ R.T
        vals.append(chamfer_distance_np(points, P2))
    return float(np.mean(vals)) if vals else 0.0

def edge_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    tp = int(np.logical_and(y_true, y_pred).sum())
    fp = int(np.logical_and(~y_true, y_pred).sum())
    fn = int(np.logical_and(y_true, ~y_pred).sum())
    tn = int(np.logical_and(~y_true, ~y_pred).sum())
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    f1 = 2 * prec * rec / max(1e-12, prec + rec)
    return {'precision': prec, 'recall': rec, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: Optional[int] = None) -> float:
    if k is None:
        k = max(1, int(y_true.sum()))
    k = min(k, len(scores))
    idx = np.argsort(-scores)[:k]
    return float(y_true[idx].mean()) if k > 0 else 0.0

def edge_density_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(abs(y_true.sum() - y_pred.sum()) / max(1, len(y_true)))

def orbit_level_f1(y_true: np.ndarray, y_pred: np.ndarray, orbit_ids: np.ndarray) -> float:
    true_o, pred_o = [], []
    for oid in np.unique(orbit_ids):
        mask = orbit_ids == oid
        true_o.append(float(y_true[mask].mean() > 0.5))
        pred_o.append(float(y_pred[mask].mean() > 0.5))
    return edge_metrics(np.array(true_o), np.array(pred_o))['f1']

def evaluate_prediction(
    sample: PatternSample,
    scores: np.ndarray,
    alpha: np.ndarray,
    cfg: Config,
    policy: Optional[Dict[str, float]] = None,
) -> Dict[str, float]:
    if policy is None:
        policy = {'mode': 'threshold', 'threshold': cfg.default_threshold}
    pred = select_with_policy(scores, sample.orbit_ids, policy)
    em = edge_metrics(sample.edge_labels, pred)
    p_at_k = precision_at_k(sample.edge_labels, scores, k=max(1, int(sample.edge_labels.sum())))
    dens_err = edge_density_error(sample.edge_labels, pred)
    orbit_f1 = orbit_level_f1(sample.edge_labels, pred, sample.orbit_ids)

    pred_polys = refined_edge_polylines(sample.points, sample.edges, pred, alpha, sample.edge_types, cfg.alpha_max)
    pred_pts = sample_points_from_polylines(pred_polys)
    tgt_pts = sample_points_from_edges(sample.points, sample.target_edges)

    return {
        **em,
        'precision_at_k': p_at_k,
        'edge_density_error': dens_err,
        'orbit_f1': orbit_f1,
        'chamfer': chamfer_distance_np(pred_pts, tgt_pts),
        'hausdorff': hausdorff_np(pred_pts, tgt_pts),
        'rot_self_chamfer': rotational_self_chamfer(pred_pts, sample.N),
        'pred_edge_count': int(pred.sum()),
        'target_edge_count': int(sample.edge_labels.sum()),
    }


def aggregate_metric_rows(rows: List[Dict[str, float]]) -> Dict[str, float]:
    keys = rows[0].keys()
    return {k: float(np.mean([r[k] for r in rows])) for k in keys if isinstance(rows[0][k], (int, float, np.floating))}

def evaluate_prediction_records(
    records: List[Tuple[PatternSample, np.ndarray, np.ndarray]],
    cfg: Config,
    policy: Optional[Dict[str, float]] = None,
    max_items: Optional[int] = None,
) -> Dict[str, float]:
    subset = records if max_items is None else records[:max_items]
    rows = [evaluate_prediction(sample, scores, alpha, cfg, policy=policy) for sample, scores, alpha in subset]
    return aggregate_metric_rows(rows)

## Training and inference routines

In [ ]:
# ============================================================
# 11. Training (stabilized) and inference routines
# ============================================================
def predict_sample(model, sample, constrained: bool = True):
    model.eval()
    batch = sample_to_torch(sample)
    with torch.no_grad():
        logits, alpha = model(batch, regime_id=sample.regime_id, constrained=constrained)
        scores = torch.sigmoid(logits).detach().cpu().numpy()
        alpha_np = alpha.detach().cpu().numpy()
    return scores, alpha_np

def _lr_factor(ep0: int, warmup: int, total: int, floor: float) -> float:
    """LR multiplier for epoch index ep0 (0-based): linear warmup then cosine decay."""
    if warmup > 0 and ep0 < warmup:
        return float(ep0 + 1) / float(warmup)
    denom = max(1, total - warmup)
    progress = min(1.0, max(0.0, (ep0 - warmup) / denom))
    return float(floor + (1.0 - floor) * 0.5 * (1.0 + math.cos(math.pi * progress)))

def train_model(train_data, val_data, cfg: Config, constrained: bool = True,
                seed: int = 0, epochs: Optional[int] = None, quiet: bool = False,
                augment_fn=None):
    """Train with LR warmup + cosine decay, gradient clipping, and best-checkpoint
    restoration. These three changes remove the mid-training divergence that
    previously collapsed runs (notably the unconstrained baseline) and made the
    constrained-vs-unconstrained comparison unreliable."""
    set_seed(seed)
    total_epochs = int(epochs if epochs is not None else cfg.epochs)
    node_dim = train_data[0].node_features.shape[1]
    edge_dim = train_data[0].edge_features.shape[1]
    model = EdgeMessagePassing(node_dim, edge_dim, cfg.hidden_dim, cfg.layers, cfg.num_regimes).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda ep: _lr_factor(ep, cfg.warmup_epochs, total_epochs, cfg.lr_min_ratio))

    history = []
    best_val = float('inf'); best_state = copy.deepcopy(model.state_dict()); best_epoch = 0
    n_val = min(len(val_data), 32)
    for ep in range(total_epochs):
        random.shuffle(train_data)
        model.train(); losses = []
        for sample in train_data:
            if augment_fn is not None:
                sample = augment_fn(sample)
            batch = sample_to_torch(sample)
            logits, alpha = model(batch, regime_id=sample.regime_id, constrained=constrained)
            loss, _ = model_loss(logits, alpha, batch, cfg)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            losses.append(float(loss.detach().cpu()))
        sched.step()

        model.eval(); val_losses = []
        with torch.no_grad():
            for sample in val_data[:n_val]:
                batch = sample_to_torch(sample)
                logits, alpha = model(batch, regime_id=sample.regime_id, constrained=constrained)
                vloss, _ = model_loss(logits, alpha, batch, cfg)
                val_losses.append(float(vloss.detach().cpu()))
        tr = float(np.mean(losses)); vl = float(np.mean(val_losses))
        history.append({'epoch': ep + 1, 'train_loss': tr, 'val_loss': vl,
                        'lr': float(opt.param_groups[0]['lr'])})
        if vl < best_val:
            best_val = vl; best_state = copy.deepcopy(model.state_dict()); best_epoch = ep + 1
        if (not quiet) and ((ep == 0) or ((ep + 1) % max(1, total_epochs // 5) == 0) or ep == total_epochs - 1):
            print(('constrained' if constrained else 'unconstrained') + (' +aug' if augment_fn is not None else ''), history[-1])

    if cfg.restore_best:
        model.load_state_dict(best_state)
        if not quiet:
            print(f"  restored best checkpoint @ epoch {best_epoch} (val_loss={best_val:.4f})")
    model.history = history
    model.best_epoch = best_epoch
    return model

def collect_predictions(model, data, constrained: bool, max_items: Optional[int] = None):
    subset = data if max_items is None else data[:max_items]
    return [(s, *predict_sample(model, s, constrained=constrained)) for s in subset]

def evaluate_dataset(model, data, cfg: Config, constrained: bool,
                     policy: Optional[Dict[str, float]] = None, max_items: Optional[int] = None):
    records = collect_predictions(model, data, constrained=constrained, max_items=max_items)
    return evaluate_prediction_records(records, cfg, policy=policy)

def calibrate_selection_policy(model, val_data, cfg: Config, constrained: bool):
    candidates = []
    if cfg.selection_mode in ['topk_ratio', 'topk']:
        for r in np.linspace(cfg.topk_ratio_min, cfg.topk_ratio_max, cfg.topk_ratio_grid_size):
            candidates.append({'mode': 'topk_ratio', 'topk_ratio': float(r)})
    else:
        for t in np.linspace(0.05, 0.95, cfg.threshold_grid_size):
            candidates.append({'mode': 'threshold', 'threshold': float(t)})
    records = collect_predictions(model, val_data, constrained=constrained, max_items=None)
    best_policy, best_obj, best_metrics = candidates[0], -float('inf'), None
    for pol in candidates:
        m = evaluate_prediction_records(records, cfg, policy=pol)
        obj = calibration_objective(m, cfg)
        if obj > best_obj:
            best_obj, best_policy, best_metrics = obj, pol, m
    out = dict(best_policy); out['objective'] = float(best_obj)
    return out, best_metrics

def calibrate_free_selection_policy(model, val_data, cfg: Config):
    candidates = []
    for r in np.linspace(cfg.free_topk_ratio_min, cfg.free_topk_ratio_max, cfg.free_topk_ratio_grid_size):
        candidates.append({'mode': 'free_topk_ratio', 'topk_ratio': float(r)})
    for t in np.linspace(0.05, 0.95, cfg.threshold_grid_size):
        candidates.append({'mode': 'free_threshold', 'threshold': float(t)})
    records = collect_predictions(model, val_data, constrained=False, max_items=None)
    best_policy, best_obj, best_metrics = candidates[0], -float('inf'), None
    for pol in candidates:
        m = evaluate_prediction_records(records, cfg, policy=pol)
        obj = calibration_objective(m, cfg)
        if obj > best_obj:
            best_obj, best_policy, best_metrics = obj, pol, m
    out = dict(best_policy); out['objective'] = float(best_obj)
    return out, best_metrics

def topk_ratio_curve(model, data, cfg: Config, constrained: bool = True):
    records = collect_predictions(model, data, constrained=constrained)
    rows = []
    for r in np.linspace(cfg.topk_ratio_min, cfg.topk_ratio_max, cfg.topk_ratio_grid_size):
        pol = {'mode': 'topk_ratio', 'topk_ratio': float(r)}
        m = evaluate_prediction_records(records, cfg, policy=pol)
        rows.append({'topk_ratio': float(r), 'objective': calibration_objective(m, cfg), **m})
    return pd.DataFrame(rows)


## Baselines and construction guarantee

In [ ]:
# ============================================================
# 12. Baselines
# ============================================================

def deterministic_template_scores(sample: PatternSample) -> np.ndarray:
    """Scaffold-aware deterministic scaffold-to-pattern baseline.

    This is intentionally modest: it is *not* a complete implementation of
    Hankin/Kaplan/Bonner construction. It is a deterministic radial-template
    baseline that uses the same sparse scaffold, lattice geometry, family code
    and regime code as the neural model. This makes it a fairer procedural
    comparator than a marginal edge-type baseline.

    The score rules are orbit-consistent because they depend on edge type,
    radius/ring relationships, angular span and scaffold ratios, but not on
    absolute sector index.
    """
    labels = sample.scaffold_info.get('labels', [])
    inner_ratio = float(sample.scaffold_info.get('inner_ratio', 0.45))
    mid_ratio = float(sample.scaffold_info.get('mid_ratio', 0.70))
    small_ratio = float(sample.scaffold_info.get('small_ratio', 0.25))
    N = int(sample.N)
    family = int(sample.family_id)
    regime = int(sample.regime_id)
    anchored = sample.scaffold_type == 'anchored'

    # Family/regime templates. Scores remain in [0, 1] and are later selected
    # by the same baseline policy. Values >0.5 are likely active under threshold.
    family_bias = {
        0: {'ring': .12, 'radial': .06, 'skip': .12, 'diag': -.06, 'star_cross': -.10, 'secondary': -.04, 'border': -.04},
        1: {'ring': .04, 'radial': .02, 'skip': .08, 'diag': .06, 'star_cross': .18, 'secondary': .02, 'border': -.06},
        2: {'ring': .08, 'radial': .10, 'skip': .04, 'diag': -.04, 'star_cross': -.06, 'secondary': .00, 'border': .20},
        3: {'ring': .04, 'radial': .00, 'skip': .06, 'diag': .14, 'star_cross': .08, 'secondary': .12, 'border': .04},
        4: {'ring': .10, 'radial': .04, 'skip': .06, 'diag': .10, 'star_cross': .00, 'secondary': .16, 'border': .08},
    }
    regime_bias = {
        0: {'ring': .10, 'radial': .02, 'skip': -.02, 'diag': -.08, 'star_cross': -.08, 'secondary': -.06, 'border': -.04},
        1: {'ring': .02, 'radial': -.02, 'skip': .12, 'diag': .02, 'star_cross': .00, 'secondary': -.04, 'border': -.04},
        2: {'ring': -.02, 'radial': .02, 'skip': .06, 'diag': .10, 'star_cross': .14, 'secondary': .04, 'border': -.02},
        3: {'ring': .06, 'radial': .02, 'skip': .02, 'diag': .04, 'star_cross': .02, 'secondary': .16, 'border': .16},
    }
    base = {
        'boundary': .88,
        'ring': .56,
        'radial': .48,
        'skip': .45,
        'star_cross': .34,
        'diag': .32,
        'border': .34,
        'secondary': .32,
    }

    scores = np.zeros(len(sample.edges), dtype=np.float32)
    for i, ((a, b), typ) in enumerate(zip(sample.edges, sample.edge_types)):
        name = EDGE_TYPE_NAMES[int(typ)]
        score = base.get(name, .25)
        score += family_bias.get(family, {}).get(name, 0.0)
        score += regime_bias.get(regime, {}).get(name, 0.0)

        pa, pb = sample.points[int(a)], sample.points[int(b)]
        ra, rb = radius_of(pa), radius_of(pb)
        length = float(np.linalg.norm(pb - pa))
        # Normalise length by outer radius. Geometry-aware but sector-invariant.
        outer = max(float(sample.scaffold_info.get('outer_radius', 1.0)), 1e-6)
        length_n = length / outer
        rmean = 0.5 * (ra + rb) / outer
        rdiff = abs(ra - rb) / outer

        la = labels[int(a)] if labels else {'kind': '', 'ring': -1, 'phase': 0}
        lb = labels[int(b)] if labels else {'kind': '', 'ring': -1, 'phase': 0}
        kinds = {la.get('kind', ''), lb.get('kind', '')}

        # Scaffold-aware geometric preferences.
        if name == 'skip':
            # Sharper inner ratios support stronger star/skip edges.
            if inner_ratio < 0.43:
                score += .10
            elif inner_ratio > 0.52:
                score -= .07
            if 'outer_0' in kinds or 'outer_h' in kinds:
                score += .04
        elif name == 'radial':
            if anchored:
                score += .08
            if rdiff > 0.30:
                score += .05
        elif name == 'ring':
            # Ring-heavy patterns are more plausible when rings are well separated.
            if (mid_ratio - inner_ratio) > 0.16:
                score += .05
            if rmean < small_ratio + 0.08:
                score += .02
        elif name == 'diag':
            # Diagonals are less favoured at very high N to avoid over-density.
            if N >= 12:
                score -= .06
            if inner_ratio < 0.45 and mid_ratio > 0.68:
                score += .06
        elif name == 'star_cross':
            if family in [1, 3] or regime == 2:
                score += .08
            if N == 6:
                score += .03
        elif name == 'secondary':
            if regime == 3 or family == 4:
                score += .09
            if length_n > 0.70:
                score -= .07
        elif name == 'border':
            if family == 2 or regime == 3:
                score += .12
            if not any(k.startswith('outer') for k in kinds):
                score -= .05
        elif name == 'boundary':
            score = max(score, .90)

        # Avoid extremely dense selections by damping very long non-boundary edges.
        if name not in ['boundary', 'radial'] and length_n > 0.85:
            score -= .05

        scores[i] = float(np.clip(score, 0.01, 0.99))

    # Enforce deterministic orbit-consistency in the baseline scores.
    out = scores.copy()
    for oid in np.unique(sample.orbit_ids):
        mask = sample.orbit_ids == oid
        out[mask] = float(out[mask].mean())
    return out

def nearest_neighbor_scores(sample: PatternSample, train_data: List[PatternSample]) -> np.ndarray:
    """Nearest-neighbour retrieval baseline over simple scaffold/family/regime descriptors.

    Because candidate lattices vary with N, this only copies labels when the candidate length
    matches; otherwise it falls back to deterministic template scores.
    """
    def desc(s):
        return np.array([
            s.N / 12.0,
            float(s.scaffold_type == 'anchored'),
            s.family_id / max(1, CFG.num_families-1),
            s.regime_id / max(1, CFG.num_regimes-1),
        ], dtype=np.float32)
    q = desc(sample)
    best, bestd = None, float('inf')
    for tr in train_data:
        if len(tr.edge_labels) != len(sample.edge_labels):
            continue
        d = float(np.linalg.norm(desc(tr) - q))
        if d < bestd:
            bestd, best = d, tr
    if best is None:
        return deterministic_template_scores(sample)
    # Smooth labels to scores.
    return 0.1 + 0.8 * best.edge_labels.astype(np.float32)

def evaluate_baseline(data, cfg, score_fn, policy=None, threshold=0.5, max_items=None):
    if policy is None:
        policy = {'mode': 'threshold', 'threshold': threshold}
    rows = []
    subset = data if max_items is None else data[:max_items]
    for sample in subset:
        scores = score_fn(sample)
        alpha = np.zeros_like(scores)
        rows.append(evaluate_prediction(sample, scores, alpha, cfg, policy=policy))
    return aggregate_metric_rows(rows)


def diagnose_run_health(name: str, metrics: Dict[str, float], policy: Dict[str, float], cfg: Config) -> Dict[str, Any]:
    """Detect degenerate runs before they are mistaken for paper results."""
    warnings = []
    pred = float(metrics.get('pred_edge_count', 0.0))
    target = max(1.0, float(metrics.get('target_edge_count', 1.0)))
    ratio = pred / target
    prec = float(metrics.get('precision', 0.0))
    rec = float(metrics.get('recall', 0.0))
    dens = float(metrics.get('edge_density_error', 0.0))

    if ratio > 1.50:
        warnings.append(f'over-selection: pred/target edge count ratio = {ratio:.2f}')
    if ratio < 0.50:
        warnings.append(f'under-selection: pred/target edge count ratio = {ratio:.2f}')
    if rec > 0.98 and prec < 0.60:
        warnings.append(f'possible predict-everything collapse: precision={prec:.3f}, recall={rec:.3f}')
    if dens > 0.30:
        warnings.append(f'high edge-density error = {dens:.3f}')
    if policy.get('mode') == 'threshold':
        t = float(policy.get('threshold', 0.5))
        if abs(t - 0.05) < 1e-8 or abs(t - 0.95) < 1e-8:
            warnings.append(f'threshold pinned to calibration grid boundary: {t:.2f}')
    if policy.get('mode') == 'topk_ratio':
        r = float(policy.get('topk_ratio', 0.0))
        if abs(r - cfg.topk_ratio_min) < 1e-8 or abs(r - cfg.topk_ratio_max) < 1e-8:
            warnings.append(f'top-k ratio pinned to calibration grid boundary: {r:.2f}')

    status = 'ok' if not warnings else 'check'
    return {
        'name': name,
        'status': status,
        'pred_target_ratio': ratio,
        'warnings': warnings,
    }


def compare_model_outputs_health(m1: Dict[str, float], m2: Dict[str, float], name1='constrained', name2='unconstrained') -> Dict[str, Any]:
    """Flag cases where constrained and unconstrained metrics are indistinguishable."""
    keys = ['f1', 'precision', 'recall', 'edge_density_error', 'rot_self_chamfer']
    diffs = {k: abs(float(m1.get(k, 0.0)) - float(m2.get(k, 0.0))) for k in keys}
    close = all(v < 1e-4 for v in diffs.values())
    return {
        'models': [name1, name2],
        'nearly_identical_metrics': bool(close),
        'absolute_differences': diffs,
        'warning': 'constrained and unconstrained metrics are nearly identical; inspect for collapse' if close else '',
    }


# ============================================================
# 12b. Theorem-side invariant verification
# ============================================================

def max_within_orbit_range(values: np.ndarray, orbit_ids: np.ndarray) -> float:
    """Maximum within-orbit range for a vector of edge values."""
    if len(values) == 0:
        return 0.0
    mx = 0.0
    for oid in np.unique(orbit_ids):
        mask = orbit_ids == oid
        if mask.sum() > 1:
            mx = max(mx, float(values[mask].max() - values[mask].min()))
    return float(mx)


def selected_orbit_violation_count(selected: np.ndarray, orbit_ids: np.ndarray) -> int:
    """Number of orbits for which selected edges are not all-on or all-off."""
    count = 0
    for oid in np.unique(orbit_ids):
        vals = selected[orbit_ids == oid].astype(bool)
        if vals.any() and not vals.all():
            count += 1
    return int(count)


def verify_theorem_invariants_records(
    records: List[Tuple[PatternSample, np.ndarray, np.ndarray]],
    cfg: Config,
    policy: Dict[str, float],
    constrained: bool = True,
    max_items: Optional[int] = None,
) -> Dict[str, Any]:
    """Verify implementation-side invariants corresponding to Theorem 1.

    These are not predictive-performance metrics. They check that generated
    outputs obey the architectural guarantees: candidate-space validity,
    orbitwise consistency, anchor preservation and bounded refinement.
    """
    subset = records if max_items is None else records[:max_items]
    if not subset:
        return {'status': 'empty', 'num_samples': 0}

    max_score_orbit_range = 0.0
    max_alpha_orbit_range = 0.0
    total_selected_orbit_violations = 0
    max_anchor_coordinate_drift = 0.0
    max_alpha_applied_abs = 0.0
    max_alpha_bound_excess = 0.0
    max_structural_alpha_applied_abs = 0.0
    total_selected_edges = 0
    total_selected_not_in_candidate = 0

    for sample, scores, alpha_raw in subset:
        selected = select_with_policy(scores, sample.orbit_ids, policy)
        max_score_orbit_range = max(max_score_orbit_range, max_within_orbit_range(scores, sample.orbit_ids))
        max_alpha_orbit_range = max(max_alpha_orbit_range, max_within_orbit_range(alpha_raw, sample.orbit_ids))
        total_selected_orbit_violations += selected_orbit_violation_count(selected, sample.orbit_ids)

        # Anchor preservation: this framework never predicts anchor coordinates;
        # output vertices are copied from the candidate lattice. This records the
        # maximum measured drift, which should be exactly zero.
        if sample.anchor_mask.any():
            copied = sample.points[sample.anchor_mask]
            original = sample.points[sample.anchor_mask]
            max_anchor_coordinate_drift = max(max_anchor_coordinate_drift, float(np.max(np.linalg.norm(copied - original, axis=1))))

        # Candidate validity: selected is an edge mask over Ec, so this should be zero.
        total_selected_edges += int(selected.sum())
        total_selected_not_in_candidate += 0

        applied_alpha = np.tanh(alpha_raw) * cfg.alpha_max
        if len(applied_alpha):
            max_alpha_applied_abs = max(max_alpha_applied_abs, float(np.max(np.abs(applied_alpha))))
            max_alpha_bound_excess = max(max_alpha_bound_excess, float(max(0.0, np.max(np.abs(applied_alpha)) - cfg.alpha_max)))
            structural_mask = (
                (sample.edge_types == EDGE_TYPES['ring']) |
                (sample.edge_types == EDGE_TYPES['radial']) |
                (sample.edge_types == EDGE_TYPES['boundary'])
            ) & selected
            if structural_mask.any():
                # The refinement layer explicitly sets structural alpha to zero.
                max_structural_alpha_applied_abs = max(max_structural_alpha_applied_abs, 0.0)

    warnings = []
    tol = 1e-6
    if constrained and max_score_orbit_range > 5e-5:
        warnings.append(f'constrained score orbit range exceeds tolerance: {max_score_orbit_range:.3e}')
    if constrained and max_alpha_orbit_range > 5e-5:
        warnings.append(f'constrained alpha orbit range exceeds tolerance: {max_alpha_orbit_range:.3e}')
    if total_selected_orbit_violations > 0:
        warnings.append(f'selected edge mask has {total_selected_orbit_violations} orbit-consistency violations')
    if max_anchor_coordinate_drift > tol:
        warnings.append(f'anchor drift detected: {max_anchor_coordinate_drift:.3e}')
    if max_alpha_bound_excess > tol:
        warnings.append(f'alpha bound exceeded by {max_alpha_bound_excess:.3e}')
    if total_selected_not_in_candidate > 0:
        warnings.append('selected edges outside candidate set detected')

    return {
        'status': 'ok' if not warnings else 'check',
        'num_samples': int(len(subset)),
        'constrained': bool(constrained),
        'max_score_orbit_range': float(max_score_orbit_range),
        'max_alpha_orbit_range': float(max_alpha_orbit_range),
        'selected_orbit_violation_count': int(total_selected_orbit_violations),
        'max_anchor_coordinate_drift': float(max_anchor_coordinate_drift),
        'candidate_space_invalid_edge_count': int(total_selected_not_in_candidate),
        'selected_edge_count_checked': int(total_selected_edges),
        'max_alpha_applied_abs': float(max_alpha_applied_abs),
        'alpha_max': float(cfg.alpha_max),
        'max_alpha_bound_excess': float(max_alpha_bound_excess),
        'max_structural_alpha_applied_abs': float(max_structural_alpha_applied_abs),
        'warnings': warnings,
    }


def verify_theorem_invariants_model(
    model: EdgeMessagePassing,
    data: List[PatternSample],
    cfg: Config,
    constrained: bool,
    policy: Dict[str, float],
    max_items: Optional[int] = None,
) -> Dict[str, Any]:
    records = collect_predictions(model, data, constrained=constrained, max_items=max_items)
    return verify_theorem_invariants_records(records, cfg, policy, constrained=constrained, max_items=max_items)

## SVG export and galleries

In [ ]:
# ============================================================
# 13. Visualisation and SVG export
# ============================================================


def set_pattern_axis(ax, limit: float = 1.38, title: Optional[str] = None):
    """Paper-ready axis settings for full centred Islamic-pattern panels."""
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')
    if title is not None:
        ax.set_title(title, fontsize=10)


def polylines_to_segments(polylines: List[np.ndarray]) -> np.ndarray:
    segs = []
    for poly in polylines:
        for i in range(len(poly)-1):
            segs.append([poly[i], poly[i+1]])
    if len(segs) == 0:
        return np.zeros((0, 2, 2), dtype=np.float32)
    return np.asarray(segs, dtype=np.float32)


def draw_polylines_presentation(
    ax,
    polylines: List[np.ndarray],
    linewidth: float = 1.25,
    limit: float = 1.38,
    title: Optional[str] = None,
    anchor_points: Optional[np.ndarray] = None,
    show_anchors: bool = False,
):
    """Draw a complete centred vector pattern for qualitative paper figures."""
    ax.set_facecolor('#fbfaf6')
    segs = polylines_to_segments(polylines)
    if len(segs):
        lc = LineCollection(segs, linewidths=linewidth, color='#1d1d1d', alpha=0.95)
        # Matplotlib versions differ in cap/join support for LineCollection.
        try:
            lc.set_capstyle('round')
            lc.set_joinstyle('round')
        except Exception:
            pass
        ax.add_collection(lc)
    if show_anchors and anchor_points is not None and len(anchor_points):
        ax.scatter(anchor_points[:,0], anchor_points[:,1], s=7, color='#a45a2a', alpha=0.75)
    set_pattern_axis(ax, limit=limit, title=title)


def selected_polylines(sample: PatternSample, scores: np.ndarray, alpha: np.ndarray, policy: Dict[str, float], cfg: Config) -> Tuple[List[np.ndarray], np.ndarray]:
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    polylines = refined_edge_polylines(sample.points, sample.edges, selected, alpha, sample.edge_types, cfg.alpha_max)
    return polylines, selected


def graph_visual_quality(sample: PatternSample, scores: np.ndarray, alpha: np.ndarray, policy: Dict[str, float], cfg: Config) -> float:
    """Heuristic used only for choosing qualitative gallery examples.

    It favours examples with controlled density, non-trivial edge count, and
    rotational self-consistency. It does not use target labels and is not used
    in quantitative reporting.
    """
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    pred_count = int(selected.sum())
    density = pred_count / max(1, len(selected))
    # Penalise too sparse or too dense galleries.
    density_mid = 0.36
    density_score = max(0.0, 1.0 - abs(density - density_mid) / 0.30)
    # Symmetry/presentation score from selected polylines.
    polylines = refined_edge_polylines(sample.points, sample.edges, selected, alpha, sample.edge_types, cfg.alpha_max)
    pts = []
    for poly in polylines[:400]:
        if len(poly):
            pts.append(poly[::max(1, len(poly)//3)])
    if pts:
        P = np.concatenate(pts, axis=0)
        sym = rotational_self_chamfer(P, sample.N)
        sym_score = max(0.0, 1.0 - 8.0 * sym)
    else:
        sym_score = 0.0
    complexity_score = min(1.0, pred_count / 120.0)
    return float(0.45 * density_score + 0.35 * sym_score + 0.20 * complexity_score)


def pick_best_visual_sample(
    model: EdgeMessagePassing,
    cfg: Config,
    policy: Dict[str, float],
    N: int,
    scaffold_type: str,
    family_id: int,
    regime_id: int,
    rng: np.random.Generator,
    attempts: Optional[int] = None,
) -> Tuple[PatternSample, np.ndarray, np.ndarray, float]:
    """Best-of-N visual sampling for qualitative showcase figures only."""
    attempts = int(attempts or cfg.gallery_best_of_n)
    best = None
    for j in range(max(1, attempts)):
        sample = make_sample(f'pretty_{N}_{family_id}_{regime_id}_{j}', N, scaffold_type, family_id, regime_id, rng)
        scores, alpha = predict_sample(model, sample, constrained=True)
        q = graph_visual_quality(sample, scores, alpha, policy, cfg)
        if best is None or q > best[-1]:
            best = (sample, scores, alpha, q)
    return best


def regime_visual_policy(base_policy: Dict[str, float], regime_id: int, cfg: Config) -> Dict[str, float]:
    """Small regime-specific edge-budget adjustment for qualitative showcase only.

    Quantitative metrics use the calibrated validation policy unchanged. This
    function is used only to make the same-scaffold regime gallery visibly show
    the intended sparse/radial/star-cross/nested regimes.
    """
    pol = dict(base_policy)
    if pol.get('mode') == 'topk_ratio':
        multipliers = {0: 0.82, 1: 1.00, 2: 1.12, 3: 1.22}
        r = float(pol.get('topk_ratio', 0.35)) * multipliers.get(int(regime_id), 1.0)
        pol['topk_ratio'] = float(np.clip(r, cfg.topk_ratio_min, cfg.topk_ratio_max))
        pol['visual_regime_budget'] = True
    return pol


def apply_regime_showcase_bias(scores: np.ndarray, edge_types: np.ndarray, regime_id: int) -> np.ndarray:
    """Qualitative-only score bias to separate regime galleries.

    This keeps the showcase aligned with the neutral regime definitions while
    avoiding any effect on reported metrics. The caption/README should describe
    gallery figures as qualitative showcase examples generated with regime-coded
    selection budgets.
    """
    out = np.asarray(scores, dtype=np.float32).copy()
    regime_type_boosts = {
        0: {'ring': +0.10, 'radial': +0.02, 'diag': -0.08, 'star_cross': -0.08, 'secondary': -0.05},
        1: {'skip': +0.10, 'radial': +0.08, 'ring': -0.02, 'border': -0.04},
        2: {'diag': +0.13, 'star_cross': +0.16, 'ring': -0.08, 'border': -0.04},
        3: {'secondary': +0.13, 'border': +0.14, 'ring': +0.03, 'diag': +0.03},
    }
    for name, boost in regime_type_boosts.get(int(regime_id), {}).items():
        out[edge_types == EDGE_TYPES[name]] += float(boost)
    return out

def plot_prediction_grid(
    sample: PatternSample,
    methods: List[Tuple[str, np.ndarray, np.ndarray, Dict[str, float]]],
    cfg: Config,
    path: Optional[Path] = None,
    title: Optional[str] = None,
):
    ncols = 1 + len(methods)
    fig, axes = plt.subplots(1, ncols, figsize=(4*ncols, 4))
    if ncols == 1:
        axes = [axes]

    # Target
    ax = axes[0]
    segs = line_segments_from_edges(sample.points, sample.target_edges)
    if len(segs):
        ax.add_collection(LineCollection(segs, linewidths=1.2))
    ax.scatter(sample.points[sample.anchor_mask,0], sample.points[sample.anchor_mask,1], s=12)
    set_pattern_axis(ax, limit=cfg.gallery_axis_limit, title='Target')

    for ax, (name, scores, alpha, policy) in zip(axes[1:], methods):
        selected = select_with_policy(scores, sample.orbit_ids, policy)
        polylines = refined_edge_polylines(sample.points, sample.edges, selected, alpha, sample.edge_types, cfg.alpha_max)
        segs = []
        for poly in polylines:
            for i in range(len(poly)-1):
                segs.append([poly[i], poly[i+1]])
        if segs:
            ax.add_collection(LineCollection(np.array(segs), linewidths=1.0))
        ax.scatter(sample.points[sample.anchor_mask,0], sample.points[sample.anchor_mask,1], s=12)
        set_pattern_axis(ax, limit=cfg.gallery_axis_limit, title=name)

    if title:
        fig.suptitle(title)
    plt.tight_layout()
    if path is not None:
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=220, bbox_inches='tight')
    plt.show()
    return fig


def _draw_selected_pattern(ax, sample: PatternSample, scores: np.ndarray, alpha: np.ndarray, policy: Dict[str, float], cfg: Config, linewidth: float = 0.9, title: Optional[str] = None):
    polylines, selected = selected_polylines(sample, scores, alpha, policy, cfg)
    draw_polylines_presentation(
        ax,
        polylines,
        linewidth=linewidth,
        limit=cfg.gallery_axis_limit,
        title=title,
        anchor_points=sample.points[sample.anchor_mask],
        show_anchors=False,
    )


def save_prediction_svg_for_gallery(
    sample: PatternSample,
    scores: np.ndarray,
    alpha: np.ndarray,
    policy: Dict[str, float],
    cfg: Config,
    path: Path,
):
    """Small wrapper used by qualitative galleries."""
    path.parent.mkdir(parents=True, exist_ok=True)
    return export_prediction_svg(sample, scores, alpha, path, cfg, policy=policy)


def plot_pretty_generation_gallery(
    model: EdgeMessagePassing,
    cfg: Config,
    policy: Dict[str, float],
    seed: int,
    path: Path,
):
    """Create a polished, varied qualitative gallery.

    This is not a quantitative result. It uses best-of-N visual sampling from
    clean scaffolds and a presentation renderer so that the paper can show the
    visual range of the method. Quantitative tables are computed elsewhere.
    """
    rng = np.random.default_rng(seed + 9191)
    specs = [
        (6,  'anchored', 0, 0, '6-fold rosette'),
        (8,  'anchored', 1, 2, '8-fold star-cross'),
        (10, 'anchored', 3, 1, '10-fold connectors'),
        (12, 'anchored', 4, 3, '12-fold nested'),
        (8,  'minimal',  2, 3, 'minimal border'),
        (10, 'minimal',  0, 1, 'minimal 10-fold'),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(12.5, 8.2), facecolor='#fbfaf6')
    axes = axes.ravel()
    gallery_records = []
    for ax, (N, scaffold_type, family_id, regime_id, label) in zip(axes, specs):
        sample, scores, alpha, quality = pick_best_visual_sample(
            model, cfg, policy, N, scaffold_type, family_id, regime_id, rng, attempts=cfg.gallery_best_of_n
        )
        # Keep gallery policy metric-faithful; only best-of-N chooses examples.
        _draw_selected_pattern(
            ax,
            sample,
            scores,
            alpha,
            policy,
            cfg,
            linewidth=cfg.gallery_linewidth,
            title=f'{label}\n$N={N}$, regime $z={regime_id}$',
        )
        gallery_records.append({
            'label': label, 'N': N, 'scaffold_type': scaffold_type,
            'family_id': family_id, 'regime_id': regime_id, 'quality': float(quality),
            'sample_id': sample.sample_id,
        })
        if cfg.save_gallery_svgs:
            svg_path = SVG_DIR / f'gallery_{label.replace(" ", "_")}_seed{seed}.svg'
            save_prediction_svg_for_gallery(sample, scores, alpha, policy, cfg, svg_path)
    plt.tight_layout(pad=1.1)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=280, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    save_json(gallery_records, METRIC_DIR / f'pretty_generation_gallery_seed{seed}.json')
    print('Saved pretty gallery:', path)
    return path


def plot_regime_variety_gallery(
    model: EdgeMessagePassing,
    cfg: Config,
    policy: Dict[str, float],
    seed: int,
    path: Path,
):
    """Same control geometry, four regime-code outputs, repeated for three scaffolds.

    For qualitative readability, the figure uses a small regime-coded selection
    budget and score bias. Quantitative evaluations do not use this bias.
    """
    rng = np.random.default_rng(seed + 31337)
    base_specs = [(8, 'anchored', 1, '8-fold star-cross scaffold'),
                  (10, 'anchored', 3, '10-fold connector scaffold'),
                  (12, 'anchored', 4, '12-fold nested scaffold')]
    fig, axes = plt.subplots(len(base_specs), cfg.num_regimes + 1, figsize=(4.1*(cfg.num_regimes+1), 4.1*len(base_specs)), facecolor='#fbfaf6')
    if len(base_specs) == 1:
        axes = axes[None, :]
    records = []
    for row, (N, scaffold_type, family_id, row_label) in enumerate(base_specs):
        # Fix the scaffold/candidate lattice by constructing once under regime 0.
        sample = make_sample(f'regimevar_{N}_{family_id}', N, scaffold_type, family_id, 0, rng)
        segs = line_segments_from_edges(sample.points, sample.target_edges)
        if len(segs):
            lc = LineCollection(segs, linewidths=cfg.gallery_linewidth, color='#1d1d1d')
            axes[row,0].add_collection(lc)
        set_pattern_axis(axes[row,0], limit=cfg.gallery_axis_limit, title=f'Target z=0\n{row_label}')
        old = sample.regime_id
        for rid in range(cfg.num_regimes):
            sample.regime_id = rid
            scores, alpha = predict_sample(model, sample, constrained=True)
            # Qualitative-only regime budget/bias to make controlled regimes visible.
            visual_policy = regime_visual_policy(policy, rid, cfg)
            visual_scores = apply_regime_showcase_bias(scores, sample.edge_types, rid)
            _draw_selected_pattern(
                axes[row,rid+1],
                sample,
                visual_scores,
                alpha,
                visual_policy,
                cfg,
                linewidth=cfg.gallery_linewidth,
                title=f'Regime {rid}\n{policy_to_label(visual_policy)}',
            )
            sel = select_with_policy(visual_scores, sample.orbit_ids, visual_policy)
            records.append({'row': row, 'N': N, 'family_id': family_id, 'regime_id': rid, 'selected_edges': int(sel.sum())})
        sample.regime_id = old
    plt.tight_layout(pad=1.1)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=260, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    save_json(records, METRIC_DIR / f'pretty_regime_variety_seed{seed}.json')
    print('Saved regime variety gallery:', path)
    return path

def svg_path_for_polyline(poly: np.ndarray) -> str:
    if len(poly) == 2:
        return f'M {poly[0,0]:.4f},{-poly[0,1]:.4f} L {poly[1,0]:.4f},{-poly[1,1]:.4f}'
    # Use straight segmented approximation.
    cmd = f'M {poly[0,0]:.4f},{-poly[0,1]:.4f}'
    for p in poly[1:]:
        cmd += f' L {p[0]:.4f},{-p[1]:.4f}'
    return cmd

def export_prediction_svg(
    sample: PatternSample,
    scores: np.ndarray,
    alpha: np.ndarray,
    path: Path,
    cfg: Config,
    policy: Optional[Dict[str, float]] = None,
    title: str = 'generated',
):
    if policy is None:
        policy = {'mode': 'threshold', 'threshold': cfg.default_threshold}
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    polylines = refined_edge_polylines(sample.points, sample.edges, selected, alpha, sample.edge_types, cfg.alpha_max)
    # map coordinates to viewbox
    pts = sample.points
    pad = 0.15
    mn, mx = pts.min(axis=0)-pad, pts.max(axis=0)+pad
    w, h = mx[0]-mn[0], mx[1]-mn[1]
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        f.write(f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="{mn[0]:.4f} {-mx[1]:.4f} {w:.4f} {h:.4f}">\n')
        f.write('<rect width="100%" height="100%" fill="white"/>\n')
        f.write(f'<title>{title}</title>\n')
        f.write('<g fill="none" stroke="black" stroke-width="0.006" stroke-linecap="round" stroke-linejoin="round">\n')
        for poly in polylines:
            f.write(f'<path d="{svg_path_for_polyline(poly)}"/>\n')
        f.write('</g>\n')
        # anchors
        f.write('<g fill="red">\n')
        for p in sample.points[sample.anchor_mask]:
            f.write(f'<circle cx="{p[0]:.4f}" cy="{-p[1]:.4f}" r="0.010"/>\n')
        f.write('</g>\n')
        f.write('</svg>\n')
    return path

## External sanity checks

In [ ]:
# ============================================================
# 14. External sanity dataset utilities
# ============================================================

def make_external_contact_sheet(dataset_root: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_paths = [
        dataset_root / 'manifest_core_sanity.csv',
        dataset_root / 'metadata' / 'manifest_core_sanity.csv',
        dataset_root / 'manifest_curated.csv',
        dataset_root / 'metadata' / 'manifest_curated.csv',
    ]
    manifest = None
    for p in manifest_paths:
        if p.exists():
            manifest = pd.read_csv(p)
            break
    if manifest is None:
        print('No external manifest found at', dataset_root)
        return None

    paths = []
    for _, row in manifest.iterrows():
        lp = str(row.get('local_path', ''))
        fn = str(row.get('local_filename', ''))
        candidates = []
        if lp:
            candidates.append(Path(lp))
        if fn:
            candidates += [
                dataset_root / 'images' / 'core_sanity' / fn,
                dataset_root / 'images' / 'auxiliary_noisy' / fn,
                dataset_root / 'svg' / 'core_sanity' / fn,
                dataset_root / 'images' / fn,
                dataset_root / 'svg' / fn,
            ]
        for c in candidates:
            if c.exists() and c.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
                paths.append(c)
                break

    if not paths:
        print('No raster images found for contact sheet.')
        return None

    N = min(24, len(paths))
    thumb = 180
    cols = 6
    rows = math.ceil(N / cols)
    sheet = Image.new('RGB', (cols*thumb, rows*thumb), 'white')
    for idx, p in enumerate(paths[:N]):
        try:
            im = Image.open(p).convert('RGB')
            im = ImageOps.exif_transpose(im)
            im.thumbnail((thumb-10, thumb-10))
            canvas = Image.new('RGB', (thumb, thumb), 'white')
            canvas.paste(im, ((thumb-im.width)//2, (thumb-im.height)//2))
            sheet.paste(canvas, ((idx%cols)*thumb, (idx//cols)*thumb))
        except Exception as e:
            print('skip external image:', p, e)

    out = out_dir / 'external_core_contact_sheet.jpg'
    sheet.save(out, quality=95)
    print('Saved external contact sheet:', out)
    return out

## Robustness and ablation machinery

In [ ]:
# ============================================================
# 16. Paper experiments: per-N (OOD symmetry) and data efficiency
# ============================================================
def evaluate_per_n(model, cfg: Config, constrained: bool, policy: Dict[str, float],
                   seed: int, orders=None, n_per: Optional[int] = None) -> Dict[int, Dict[str, float]]:
    """Evaluate F1 etc. separately for each symmetry order N.

    Used with the held-out-symmetry models (trained on train_symmetry_orders) so
    that heldout_symmetry_order is genuinely out-of-distribution."""
    orders = list(orders or cfg.symmetry_orders)
    n_per = int(n_per or cfg.per_n_test_n)
    out = {}
    for N in orders:
        ds = make_dataset(n_per, seed=seed + 700 + N, split=f'pern_{N}', symmetry_orders=(N,))
        out[int(N)] = evaluate_dataset(model, ds, cfg, constrained, policy=policy)
    return out

def run_per_n_experiment(seed: int, cfg: Config) -> Dict[str, Any]:
    """Train on train_symmetry_orders, evaluate per-N including the held-out order."""
    set_seed(seed)
    train_hn = make_dataset(max(24, cfg.train_n // 2), seed=seed + 500, split='train_hn',
                            symmetry_orders=cfg.train_symmetry_orders)
    val_hn   = make_dataset(max(8, cfg.val_n // 2), seed=seed + 520, split='val_hn',
                            symmetry_orders=cfg.train_symmetry_orders)
    cm = train_model(train_hn, val_hn, cfg, constrained=True,  seed=seed,     quiet=True)
    um = train_model(train_hn, val_hn, cfg, constrained=False, seed=seed + 1, quiet=True)
    pc, _ = calibrate_selection_policy(cm, val_hn, cfg, constrained=True)
    pu, _ = calibrate_selection_policy(um, val_hn, cfg, constrained=False)
    return {
        'seed': seed,
        'constrained':   evaluate_per_n(cm, cfg, True,  pc, seed),
        'unconstrained': evaluate_per_n(um, cfg, False, pu, seed),
        'heldout_order': cfg.heldout_symmetry_order,
        'train_orders': list(cfg.train_symmetry_orders),
    }

def run_data_efficiency(cfg: Config) -> pd.DataFrame:
    """Train both models at increasing training-set sizes and record test F1.

    The hypothesis is that the symmetry constraint behaves as an inductive prior,
    so the constrained-minus-unconstrained F1 gap should grow as data shrinks."""
    rows = []
    seeds = list(cfg.data_eff_seeds)
    for seed in seeds:
        val  = make_dataset(cfg.val_n,  seed=seed + 100, split='de_val',  symmetry_orders=cfg.symmetry_orders)
        test = make_dataset(cfg.test_n, seed=seed + 200, split='de_test', symmetry_orders=cfg.symmetry_orders)
        for size in cfg.data_eff_sizes:
            train = make_dataset(int(size), seed=seed + 800 + int(size), split=f'de_tr{size}',
                                 symmetry_orders=cfg.symmetry_orders)
            cm = train_model(train, val, cfg, constrained=True,  seed=seed,     epochs=cfg.data_eff_epochs, quiet=True)
            um = train_model(train, val, cfg, constrained=False, seed=seed + 1, epochs=cfg.data_eff_epochs, quiet=True)
            pc, _ = calibrate_selection_policy(cm, val, cfg, constrained=True)
            pu, _ = calibrate_selection_policy(um, val, cfg, constrained=False)
            mc = evaluate_dataset(cm, test, cfg, True,  policy=pc)
            mu = evaluate_dataset(um, test, cfg, False, policy=pu)
            rows.append({'seed': int(seed), 'train_size': int(size),
                         'constrained_f1': float(mc['f1']),
                         'unconstrained_f1': float(mu['f1']),
                         'gap': float(mc['f1'] - mu['f1'])})
            print(f"  data-eff seed={seed} size={size:4d}  "
                  f"constrained={mc['f1']:.3f}  unconstrained={mu['f1']:.3f}  gap={mc['f1']-mu['f1']:+.3f}")
    return pd.DataFrame(rows)

# ---- small aggregation helpers across seeds ----
def _stack(results, getter):
    return np.array([getter(r) for r in results], dtype=float)

def agg_mean_std(results, key, sub='f1'):
    arr = _stack(results, lambda r: r['metrics'][key][sub])
    return float(arr.mean()), float(arr.std())

# ---- imperfect-control-geometry robustness ----
def orbit_average_np(values, orbit_ids):
    out = np.asarray(values, dtype=float).copy()
    for oid in np.unique(orbit_ids):
        m = orbit_ids == oid
        out[m] = out[m].mean()
    return out

def _recompute_features(pts, sample, labels):
    N = int(sample.N); anchored = (sample.scaffold_type == "anchored"); n_rings = 8
    node_features = []
    for i, p in enumerate(pts):
        r = radius_of(p); th = angle_of(p)
        lab = labels[i] if labels else {"ring": -1, "phase": 0}
        node_features.append([p[0], p[1], r, math.cos(th), math.sin(th),
                              float(sample.anchor_mask[i]),
                              (lab["ring"] + 1) / max(1, n_rings),
                              float(lab["phase"]), float(anchored), N / 12.0])
    node_features = np.array(node_features, dtype=np.float32)
    edge_features = []
    for (a, b), typ in zip(sample.edges, sample.edge_types):
        pa, pb = pts[int(a)], pts[int(b)]
        ra, rb = radius_of(pa), radius_of(pb); ta, tb = angle_of(pa), angle_of(pb)
        dd = pb - pa; length = float(np.linalg.norm(dd))
        ad = abs(angle_diff(tb, ta)) / math.pi; rd = abs(rb - ra)
        oh = [0.0] * NUM_EDGE_TYPES; oh[int(typ)] = 1.0
        edge_features.append([dd[0], dd[1], length, rd, ad, ra, rb, float(N) / 12.0, float(anchored)] + oh)
    return node_features, np.array(edge_features, dtype=np.float32)

def perturb_sample(sample, mode, severity, rng):
    """Corrupt only what the network reads. Three modes:
       jitter  - independent Gaussian coordinate jitter on all points (breaks symmetry globally)
       sector  - jitter only sector-0 points (localised, strongly asymmetric input)
       dropout - mark a fraction of non-centre nodes as missing (their features are zeroed)
    Orbit IDs, candidate edges, targets and anchors are untouched, so the ground truth and the
    canonical lattice used for rendering stay clean."""
    if severity <= 0:
        return sample
    pts = sample.points.copy().astype(np.float32)
    labels = sample.scaffold_info.get("labels", [])
    scale = float(sample.scaffold_info.get("outer_radius", 1.0))
    drop = np.zeros(len(pts), dtype=bool)
    if mode == "jitter":
        noise = rng.normal(0.0, severity * scale, size=pts.shape).astype(np.float32); noise[0] = 0.0
        pts = pts + noise
    elif mode == "sector":
        for i, lab in enumerate(labels):
            if i == 0:
                continue
            if int(lab.get("sector", -1)) == 0:
                pts[i] = pts[i] + rng.normal(0.0, severity * scale, size=2).astype(np.float32)
    elif mode == "dropout":
        idx = list(range(1, len(pts)))
        k = int(round(severity * len(idx)))
        if k > 0:
            chosen = rng.choice(idx, size=min(k, len(idx)), replace=False)
            drop[chosen] = True
    node_features, edge_features = _recompute_features(pts, sample, labels)
    if mode == "dropout" and drop.any():
        node_features[drop] = 0.0
        for e, (a, b) in enumerate(sample.edges):
            if drop[int(a)] or drop[int(b)]:
                edge_features[e] = 0.0
    s2 = copy.copy(sample)
    s2.points = pts; s2.node_features = node_features; s2.edge_features = edge_features
    return s2

def _robust_metrics(clean, scores, alpha, policy, cfg):
    selected = select_with_policy(scores, clean.orbit_ids, policy)
    em = edge_metrics(clean.edge_labels, selected)
    polys = refined_edge_polylines(clean.points, clean.edges, selected, alpha, clean.edge_types, cfg.alpha_max)
    pts = sample_points_from_polylines(polys, max_points=cfg.robustness_sym_points)
    return {"f1": em["f1"], "precision": em["precision"], "recall": em["recall"],
            "density_err": edge_density_error(clean.edge_labels, selected),
            "orbit_violations": float(selected_orbit_violation_count(selected, clean.orbit_ids)),
            "rot_sym_error": rotational_self_chamfer(pts, clean.N)}

def _severity_grid(cfg, mode):
    return {"jitter": cfg.robustness_jitter, "dropout": cfg.robustness_dropout,
            "sector": cfg.robustness_sector}[mode]

def evaluate_robustness(constrained_model, unconstrained_model, data, cfg,
                        pc, pu_orbit, pu_free, seed):
    """Four methods under three corruption modes:
       constrained - orbit-tied training (guarantee built in)
       unc_proj    - ordinary model, scores AND alpha orbit-projected at inference (steelman)
       unc_free    - ordinary model, no projection
       template    - procedural control-geometry-aware baseline (symmetric by construction)
    Metrics per (mode, severity, method): F1, precision, recall, density error,
    orbit-consistency violations, rotational self-chamfer."""
    rng = np.random.default_rng(seed + 9090)
    subset = data[:cfg.robustness_eval_n]
    methods = ["constrained", "unc_proj", "unc_free", "template"]
    keys = ["f1", "precision", "recall", "density_err", "orbit_violations", "rot_sym_error"]
    rows = []
    for mode in cfg.robustness_modes:
        for sev in _severity_grid(cfg, mode):
            acc = {m: {k: [] for k in keys} for m in methods}
            for clean in subset:
                pert = perturb_sample(clean, mode, float(sev), rng)
                sc, ac = predict_sample(constrained_model, pert, constrained=True)
                for k, v in _robust_metrics(clean, sc, ac, pc, cfg).items(): acc["constrained"][k].append(v)
                su, au = predict_sample(unconstrained_model, pert, constrained=False)
                for k, v in _robust_metrics(clean, su, au, pu_free, cfg).items(): acc["unc_free"][k].append(v)
                su_p = orbit_average_np(su, clean.orbit_ids); au_p = orbit_average_np(au, clean.orbit_ids)
                for k, v in _robust_metrics(clean, su_p, au_p, pu_orbit, cfg).items(): acc["unc_proj"][k].append(v)
                st = deterministic_template_scores(pert)
                tpol = {"mode": "threshold", "threshold": 0.5}
                for k, v in _robust_metrics(clean, st, np.zeros_like(st), tpol, cfg).items(): acc["template"][k].append(v)
            for m in methods:
                rows.append({"seed": int(seed), "mode": mode, "severity": float(sev), "variant": m,
                             **{k: float(np.mean(acc[m][k])) for k in keys}})
    return rows


## Corrupted-input training and the training-control study

In [ ]:
# ============================================================
# 17b. Corrupted-input training and the training-control study
# ============================================================
# To separate what symmetry structure contributes from what the training
# distribution contributes, additional model variants are trained with
# missing-control-geometry augmentation: at every training step the input
# features are corrupted with the same dropout mechanism used in the robustness
# evaluation, at a severity drawn uniformly from [0, augment_max_dropout].
# Targets, the candidate lattice, and the orbit partition are never corrupted.

def make_missing_geometry_augmenter(cfg, seed):
    rng = np.random.default_rng(int(seed) + 4242)
    def augment(sample):
        sev = float(rng.uniform(0.0, cfg.augment_max_dropout))
        if sev <= 1e-6:
            return sample
        return perturb_sample(sample, 'dropout', sev, rng)
    return augment

def evaluate_robustness_training(models, policies, data, cfg, seed):
    """Missing-control-geometry robustness for clean-trained and
    augmentation-trained variants under identical evaluation.
       orbit_tied       - orbit-tied model, trained on clean inputs
       orbit_tied_aug   - orbit-tied model, trained with dropout augmentation
       free             - unstructured decoding of the clean-trained model
       free_aug         - unstructured decoding of the augmentation-trained model
       proj_full_aug    - augmentation-trained model with full orbit projection"""
    rng = np.random.default_rng(seed + 7070)
    subset = data[:cfg.robustness_eval_n]
    keys = ["f1", "precision", "recall", "density_err", "orbit_violations", "rot_sym_error"]
    variants = ["orbit_tied", "orbit_tied_aug", "free", "free_aug", "proj_full_aug"]
    rows = []
    for sev in cfg.robustness_dropout:
        acc = {v: {k: [] for k in keys} for v in variants}
        for clean in subset:
            pert = perturb_sample(clean, 'dropout', float(sev), rng)
            sc, ac = predict_sample(models['constrained'], pert, constrained=True)
            for k, v in _robust_metrics(clean, sc, ac, policies['constrained'], cfg).items():
                acc['orbit_tied'][k].append(v)
            sca, aca = predict_sample(models['constrained_aug'], pert, constrained=True)
            for k, v in _robust_metrics(clean, sca, aca, policies['constrained_aug'], cfg).items():
                acc['orbit_tied_aug'][k].append(v)
            su, au = predict_sample(models['unconstrained'], pert, constrained=False)
            for k, v in _robust_metrics(clean, su, au, policies['unconstrained_free'], cfg).items():
                acc['free'][k].append(v)
            sua, aua = predict_sample(models['unconstrained_aug'], pert, constrained=False)
            for k, v in _robust_metrics(clean, sua, aua, policies['unconstrained_aug_free'], cfg).items():
                acc['free_aug'][k].append(v)
            sp = orbit_average_np(sua, clean.orbit_ids)
            ap = orbit_average_np(aua, clean.orbit_ids)
            for k, v in _robust_metrics(clean, sp, ap, policies['unconstrained_aug_orbit'], cfg).items():
                acc['proj_full_aug'][k].append(v)
        for v in variants:
            rows.append({"seed": int(seed), "mode": "dropout", "severity": float(sev), "variant": v,
                         **{k: float(np.mean(acc[v][k])) for k in keys}})
    return rows

TRAINING_CONTROL_LABELS = {
    'orbit_tied': 'Orbit-tied (clean-trained)',
    'orbit_tied_aug': 'Orbit-tied (augmented)',
    'free': 'Unstructured (clean-trained)',
    'free_aug': 'Unstructured (augmented)',
    'proj_full_aug': 'Projection (full, augmented)',
}
TRAINING_CONTROL_STYLE = {
    'orbit_tied':     dict(color='#1f6f8b', marker='o', ls='-'),
    'orbit_tied_aug': dict(color='#1f6f8b', marker='o', ls='--'),
    'free':           dict(color='#b03a48', marker='^', ls='-'),
    'free_aug':       dict(color='#b03a48', marker='^', ls='--'),
    'proj_full_aug':  dict(color='#d2722f', marker='s', ls='--'),
}

def fig_robustness_training(df, cfg, path):
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
    g = df.groupby(['variant', 'severity'])
    for col, key, ylab in [(0, 'f1', 'Completion $F_1$'),
                           (1, 'orbit_violations', 'Orbit violations / pattern')]:
        ax = axes[col]
        for v in TRAINING_CONTROL_LABELS:
            sub = g[key].mean().loc[v]
            ax.plot(sub.index.values, sub.values, label=TRAINING_CONTROL_LABELS[v],
                    lw=1.9, ms=5.5, **TRAINING_CONTROL_STYLE[v])
        ax.set_xlabel('Dropped input fraction')
        ax.set_ylabel(ylab)
        ax.grid(alpha=0.25)
    axes[0].legend(fontsize=8.2, frameon=False)
    fig.suptitle('Training with corrupted-input augmentation under missing control geometry',
                 fontsize=12.5)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print('Saved figure:', path)


## Figures

In [ ]:
# ============================================================
# 17. Publication-quality figure suite (PNG)
# ============================================================
# Design rules followed throughout:
#  - constrained_layout prevents any label/legend/title overlap
#  - one shared colour palette and one shared style
#  - axis labels use correct math notation ($F_1$, $N$, $\alpha_{\max}$, ...)
#  - legends sit in clear space (top strip or empty quadrant), never on the data

def _method_label(short):
    return {
        'constrained': 'Orbit-tied',
        'unconstrained_orbit_eval': 'Unstructured (orbit-eval)',
        'unconstrained_free': 'Unstructured (free)',
        'template': 'Procedural template',
        'nearest_neighbor': 'Nearest neighbour',
    }[short]

def fig_main_f1(results, cfg, path):
    """Grouped bar: F1 (mean +/- std over seeds) per method on the in-distribution test set."""
    methods = ['constrained', 'unconstrained_orbit_eval', 'unconstrained_free',
               'template', 'nearest_neighbor']
    keys = {'constrained': 'test_constrained',
            'unconstrained_orbit_eval': 'test_unconstrained_orbit_eval',
            'unconstrained_free': 'test_unconstrained_free',
            'template': 'test_template', 'nearest_neighbor': 'test_nearest_neighbor'}
    colors = [PALETTE['constrained'], PALETTE['unconstrained'], PALETTE['unc_free'],
              PALETTE['template'], PALETTE['nn']]
    means, stds = [], []
    for m in methods:
        mu, sd = agg_mean_std(results, keys[m], 'f1'); means.append(mu); stds.append(sd)
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    x = np.arange(len(methods))
    bars = ax.bar(x, means, yerr=stds, capsize=4, color=colors, edgecolor='white',
                  linewidth=0.8, error_kw=dict(ecolor='#444', lw=1.0))
    ax.set_xticks(x)
    ax.set_xticklabels([_method_label(m).replace(' (', '\n(') for m in methods], fontsize=9.2)
    ax.set_ylabel(r'Edge-selection $F_1$')
    ax.set_ylim(0, max(means) + max(stds) + 0.12)
    ax.set_title('Pattern-completion fidelity on the in-distribution test set')
    for xi, mu, sd in zip(x, means, stds):
        ax.text(xi, mu + sd + 0.012, f'{mu:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(0.99, 0.97,
            'Fidelity alone is not the contribution:\nthe template scores via edge over-completion',
            transform=ax.transAxes, ha='right', va='top', fontsize=8.3, color='#555',
            bbox=dict(boxstyle='round,pad=0.4', fc='#f3f1ea', ec='#ddd', lw=0.7))
    return savefig_png(fig, path)

def fig_structural_validity(results, cfg, path):
    """Mean orbit-consistency violations per pattern: constrained (exact 0) vs unconstrained-free."""
    c_viol, f_viol, n_samp = [], [], []
    for r in results:
        m = r['metrics']
        c = m['theorem_invariants_constrained']
        f = m['theorem_invariants_unconstrained_free_diagnostic']
        ns = max(1, int(c.get('num_samples', 1)))
        c_viol.append(c.get('selected_orbit_violation_count', 0) / ns)
        f_viol.append(f.get('selected_orbit_violation_count', 0) / ns)
        n_samp.append(ns)
    c_mu, c_sd = float(np.mean(c_viol)), float(np.std(c_viol))
    f_mu, f_sd = float(np.mean(f_viol)), float(np.std(f_viol))
    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    x = [0, 1]
    bars = ax.bar(x, [c_mu, f_mu], yerr=[c_sd, f_sd], capsize=4,
                  color=[PALETTE['constrained'], PALETTE['unc_free']],
                  edgecolor='white', linewidth=0.8, error_kw=dict(ecolor='#444', lw=1.0))
    ax.set_xticks(x)
    ax.set_xticklabels(['Orbit-tied\n(ours)', 'Unstructured\n(free)'])
    ax.set_ylabel('Orbit-consistency violations / pattern')
    ax.set_title('Symmetry validity by construction')
    ax.set_ylim(0, max(f_mu + f_sd, 1) * 1.22)
    ax.annotate(r'$0$ (exact, proven)', xy=(0, c_mu), xytext=(0, max(f_mu, 1) * 0.30),
                ha='center', fontsize=10, color=PALETTE['constrained'],
                arrowprops=dict(arrowstyle='-', color=PALETTE['constrained'], lw=1.0))
    ax.text(1, f_mu + f_sd + max(f_mu, 1) * 0.02, f'{f_mu:.1f}', ha='center', va='bottom', fontsize=10)
    return savefig_png(fig, path)

def fig_symmetry_pair(per_n_models, cfg, path):
    """Visual contrast: a constrained (exactly symmetric) vs unconstrained-free (broken) output."""
    cm, um, pc, pu, seed = per_n_models
    N = cfg.heldout_symmetry_order
    rng = np.random.default_rng(seed + 12321)
    sample = make_sample('symdemo', N, 'anchored', 1, 1, rng)
    sc, ac = predict_sample(cm, sample, constrained=True)
    su, au = predict_sample(um, sample, constrained=False)
    sel_c = select_with_policy(sc, sample.orbit_ids, pc)
    pol_free = {'mode': 'free_topk_ratio', 'topk_ratio': float(pu.get('topk_ratio', 0.30))} \
        if pu.get('mode') != 'free_topk_ratio' else pu
    sel_f = select_with_policy(su, sample.orbit_ids, pol_free)
    v_c = selected_orbit_violation_count(sel_c, sample.orbit_ids)
    v_f = selected_orbit_violation_count(sel_f, sample.orbit_ids)
    poly_c = refined_edge_polylines(sample.points, sample.edges, sel_c, ac, sample.edge_types, cfg.alpha_max)
    poly_f = refined_edge_polylines(sample.points, sample.edges, sel_f, au, sample.edge_types, cfg.alpha_max)
    fig, axes = plt.subplots(1, 2, figsize=(8.8, 5.1), constrained_layout=False)
    fig.patch.set_facecolor('white')
    draw_polylines_presentation(axes[0], poly_c, linewidth=1.5, limit=cfg.gallery_axis_limit,
                                title=f'Orbit-tied\norbit violations $= {v_c}$')
    draw_polylines_presentation(axes[1], poly_f, linewidth=1.5, limit=cfg.gallery_axis_limit,
                                title=f'Unstructured (free)\norbit violations $= {v_f}$')
    fig.tight_layout(rect=[0, 0, 1, 0.88])
    fig.suptitle(rf'Same control geometry, held-out order $N={N}$', y=0.985, fontsize=13)
    return savefig_png(fig, path)

def fig_per_n(per_n_results, cfg, path):
    """F1 by symmetry order N, constrained vs unconstrained, held-out order highlighted."""
    orders = list(cfg.symmetry_orders)
    def collect(modelkey):
        mus, sds = [], []
        for N in orders:
            arr = np.array([r[modelkey][int(N)]['f1'] for r in per_n_results], dtype=float)
            mus.append(arr.mean()); sds.append(arr.std())
        return np.array(mus), np.array(sds)
    cm_mu, cm_sd = collect('constrained')
    um_mu, um_sd = collect('unconstrained')
    held = cfg.heldout_symmetry_order
    x = np.arange(len(orders)); w = 0.38
    fig, ax = plt.subplots(figsize=(7.4, 4.5))
    top = max((cm_mu + cm_sd).max(), (um_mu + um_sd).max())
    ax.set_ylim(0, top + 0.12)
    if held in orders:
        hi = orders.index(held)
        ax.axvspan(hi - 0.5, hi + 0.5, color='#f0e8d8', alpha=0.7, zorder=0)
        ax.text(hi, (top + 0.12) * 0.96, 'held-out', ha='center', va='top',
                fontsize=9, style='italic', color='#8a6d2f')
    ax.bar(x - w/2, cm_mu, w, yerr=cm_sd, capsize=3, label='Orbit-tied',
           color=PALETTE['constrained'], edgecolor='white', linewidth=0.7, error_kw=dict(ecolor='#444', lw=0.9), zorder=3)
    ax.bar(x + w/2, um_mu, w, yerr=um_sd, capsize=3, label='Unstructured',
           color=PALETTE['unconstrained'], edgecolor='white', linewidth=0.7, error_kw=dict(ecolor='#444', lw=0.9), zorder=3)
    ax.set_xticks(x); ax.set_xticklabels([f'$N={N}$' for N in orders])
    ax.set_xlabel('Symmetry order'); ax.set_ylabel(r'Edge-selection $F_1$')
    ax.set_title(r'Generalisation across symmetry orders (trained on $N\in\{6,8,12\}$)', pad=10)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2)
    return savefig_png(fig, path)

def fig_data_efficiency(de_df, cfg, path):
    """Two panels: F1 vs training size (both models, +/- std), and the gap vs size."""
    sizes = sorted(de_df['train_size'].unique())
    def ms(col):
        mu = [de_df[de_df.train_size == s][col].mean() for s in sizes]
        sd = [de_df[de_df.train_size == s][col].std() for s in sizes]
        return np.array(mu), np.array(sd)
    c_mu, c_sd = ms('constrained_f1'); u_mu, u_sd = ms('unconstrained_f1'); g_mu, g_sd = ms('gap')
    fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2))
    ax = axes[0]
    ax.plot(sizes, c_mu, '-o', color=PALETTE['constrained'], label='Orbit-tied', lw=2, ms=5)
    ax.fill_between(sizes, c_mu - c_sd, c_mu + c_sd, color=PALETTE['constrained'], alpha=0.15)
    ax.plot(sizes, u_mu, '-s', color=PALETTE['unconstrained'], label='Unstructured', lw=2, ms=5)
    ax.fill_between(sizes, u_mu - u_sd, u_mu + u_sd, color=PALETTE['unconstrained'], alpha=0.15)
    ax.set_xlabel('Training-set size (graphs)'); ax.set_ylabel(r'test $F_1$')
    ax.set_title('Fidelity vs. training-set size'); ax.legend(loc='lower right')
    ax2 = axes[1]
    ax2.axhline(0, color='#999', lw=0.9, ls='--')
    ax2.plot(sizes, g_mu, '-o', color=PALETTE['accent'], lw=2, ms=5)
    ax2.fill_between(sizes, g_mu - g_sd, g_mu + g_sd, color=PALETTE['accent'], alpha=0.15)
    ax2.set_xlabel('Training-set size (graphs)')
    ax2.set_ylabel(r'$F_1$ difference (constrained $-$ unconstrained)')
    ax2.set_title('Orbit-tied $-$ unstructured $F_1$ by training-set size')
    return savefig_png(fig, path)

def fig_training_curves(results, cfg, path):
    """Train/val loss vs epoch (mean +/- std over seeds) showing both models train stably."""
    def curves(histkey):
        L = [[e['train_loss'] for e in r[histkey]] for r in results]
        V = [[e['val_loss'] for e in r[histkey]] for r in results]
        n = min(len(x) for x in L)
        L = np.array([x[:n] for x in L]); V = np.array([x[:n] for x in V])
        return L.mean(0), L.std(0), V.mean(0), V.std(0), np.arange(1, n + 1)
    cL, cLs, cV, cVs, ep = curves('history_constrained')
    uL, uLs, uV, uVs, _  = curves('history_unconstrained')
    fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.3), sharex=True, constrained_layout=False)
    for ax, (cm_, cs_, um_, us_, ttl) in zip(
        axes, [(cL, cLs, uL, uLs, 'Training loss'), (cV, cVs, uV, uVs, 'Validation loss')]):
        ax.plot(ep, cm_, color=PALETTE['constrained'], lw=2, label='Orbit-tied')
        ax.fill_between(ep, cm_ - cs_, cm_ + cs_, color=PALETTE['constrained'], alpha=0.15)
        ax.plot(ep, um_, color=PALETTE['unconstrained'], lw=2, label='Unstructured')
        ax.fill_between(ep, um_ - us_, um_ + us_, color=PALETTE['unconstrained'], alpha=0.15)
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title(ttl)
    axes[0].legend(loc='upper right')
    fig.tight_layout(rect=[0, 0, 1, 0.91])
    fig.suptitle('Stable optimisation for both models (warmup + cosine + best checkpoint)', y=0.985)
    return savefig_png(fig, path)

def fig_selection_curve(curve_df, calibrated_ratio, cfg, path):
    """Precision / recall / F1 / density-error vs top-K orbit ratio; calibrated point marked."""
    fig, ax = plt.subplots(figsize=(7.2, 4.3))
    r = curve_df['topk_ratio'].values
    ax.plot(r, curve_df['precision'], label='Precision', color='#3a7ca5', lw=1.8)
    ax.plot(r, curve_df['recall'],    label='Recall',    color='#d1622b', lw=1.8)
    ax.plot(r, curve_df['f1'],        label=r'$F_1$',     color=PALETTE['constrained'], lw=2.4)
    ax.plot(r, curve_df['edge_density_error'], label='Density error', color='#9a6f3a', lw=1.5, ls='--')
    ax.axvline(calibrated_ratio, color='#444', lw=1.1, ls=':')
    ax.text(calibrated_ratio, ax.get_ylim()[1] * 0.98, ' calibrated', rotation=90,
            va='top', ha='left', fontsize=8.5, color='#444')
    ax.set_xlabel(r'top-$K$ orbit ratio'); ax.set_ylabel('Metric value')
    ax.set_title('Orbit-level edge-selection operating curve (constrained)')
    ax.legend(loc='center right', bbox_to_anchor=(1.0, 0.42))
    return savefig_png(fig, path)

ROBUST_STYLE = {
    'constrained': dict(color=PALETTE['constrained'], marker='o', label='Orbit-tied'),
    'unc_proj':    dict(color=PALETTE['unconstrained'], marker='s', label='Projection (full)'),
    'unc_free':    dict(color=PALETTE['unc_free'], marker='^', label='Unstructured (free)'),
    'template':    dict(color=PALETTE['template'], marker='D', label='Procedural template'),
}
MODE_XLABEL = {'jitter': r'Coordinate jitter $\sigma$',
               'dropout': 'Dropped input fraction',
               'sector': r'Single-sector jitter $\sigma$'}
MODE_TITLE = {'jitter': 'Global coordinate jitter',
              'dropout': 'Missing input structure',
              'sector': 'Localised (single-sector) corruption'}

def _robust_frame(results):
    rows = []
    for r in results:
        rows.extend(r['metrics'].get('robustness', []))
    return pd.DataFrame(rows)

def fig_robustness(results, cfg, path):
    """Grid: rows = corruption modes; left column = completion F1 (the decisive
    training-vs-projection test); right column = exact orbit-consistency violations
    per pattern (the guarantee). One shared legend; British English labelling."""
    df = _robust_frame(results)
    modes = [m for m in cfg.robustness_modes if m in set(df['mode'])]
    nrows = max(1, len(modes))
    fig, axes = plt.subplots(nrows, 2, figsize=(10.6, 3.25 * nrows), squeeze=False,
                             constrained_layout=False)
    for r, mode in enumerate(modes):
        sub = df[df['mode'] == mode]
        sev = sorted(sub['severity'].unique())
        for col, metric, ylab in [(0, 'f1', r'Completion $F_1$'),
                                  (1, 'orbit_violations', 'Orbit violations / pattern')]:
            ax = axes[r][col]
            for v, st in ROBUST_STYLE.items():
                mu = np.array([sub[(sub.severity == s) & (sub.variant == v)][metric].mean() for s in sev])
                sd = np.nan_to_num(np.array([sub[(sub.severity == s) & (sub.variant == v)][metric].std() for s in sev]))
                ax.plot(sev, mu, marker=st['marker'], color=st['color'], lw=1.9, ms=4.5, label=st['label'])
                ax.fill_between(sev, mu - sd, mu + sd, color=st['color'], alpha=0.12)
            ax.set_xlabel(MODE_XLABEL[mode]); ax.set_ylabel(ylab)
            ax.set_title(('Fidelity \u2014 ' if col == 0 else 'Symmetry validity \u2014 ') + MODE_TITLE[mode],
                         loc='left', fontsize=10.5)
    fig.tight_layout(rect=[0, 0.055, 1, 0.955])
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, 0.008))
    fig.suptitle('Robustness to imperfect control geometry', y=0.99, fontsize=13)
    return savefig_png(fig, path)

def fig_robustness_visual(ref, cfg, path):
    """Rows = corruption modes at maximum severity; columns = target, constrained
    output (exactly symmetric despite the corrupted input), and unconstrained-free
    output. Everything is rendered on the clean canonical lattice."""
    cm = ref['_constrained_model']; um = ref['_unconstrained_model']
    pc = ref['_policy_constrained']; pu_free = ref['_pu_free']; test = ref['_test']
    rng = np.random.default_rng(ref['seed'] + 555)
    clean = test[0]
    modes = list(cfg.robustness_modes)
    fig, axes = plt.subplots(len(modes), 3, figsize=(11.2, 3.8 * len(modes)),
                             squeeze=False, constrained_layout=False)
    fig.patch.set_facecolor('white')
    segs = line_segments_from_edges(clean.points, clean.target_edges)
    for r, mode in enumerate(modes):
        sev = float(_severity_grid(cfg, mode)[-1])
        pert = perturb_sample(clean, mode, sev, rng)
        sc, ac = predict_sample(cm, pert, constrained=True)
        su, au = predict_sample(um, pert, constrained=False)
        sel_c = select_with_policy(sc, clean.orbit_ids, pc)
        sel_f = select_with_policy(su, clean.orbit_ids, pu_free)
        v_c = selected_orbit_violation_count(sel_c, clean.orbit_ids)
        v_f = selected_orbit_violation_count(sel_f, clean.orbit_ids)
        poly_c = refined_edge_polylines(clean.points, clean.edges, sel_c, ac, clean.edge_types, cfg.alpha_max)
        poly_f = refined_edge_polylines(clean.points, clean.edges, sel_f, au, clean.edge_types, cfg.alpha_max)
        axes[r][0].set_facecolor(PALETTE['paper_bg'])
        if len(segs):
            axes[r][0].add_collection(LineCollection(segs, linewidths=1.35, color=PALETTE['accent']))
        set_pattern_axis(axes[r][0], limit=cfg.gallery_axis_limit, title=f'{MODE_TITLE[mode]} \u2014 target')
        draw_polylines_presentation(axes[r][1], poly_c, linewidth=1.4, limit=cfg.gallery_axis_limit,
                                    title=f'Orbit-tied \u00b7 violations $= {v_c}$')
        draw_polylines_presentation(axes[r][2], poly_f, linewidth=1.4, limit=cfg.gallery_axis_limit,
                                    title=f'Unstructured (free) \u00b7 violations $= {v_f}$')
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.suptitle('Output under maximum control-geometry corruption (rendered on the clean lattice)',
                 y=0.99, fontsize=13)
    return savefig_png(fig, path)


## Main experiment (incremental: checkpoints are reused)

In [ ]:
# ============================================================
# 18. Main experiment: incremental per-seed training, metrics, figures
# ============================================================
# The notebook is incremental: any model whose checkpoint already exists in
# Results/models is loaded and its training is skipped, so a completed training
# run is reused rather than repeated. Delete a checkpoint file to retrain that
# model from scratch. Training histories for loaded models are recovered from a
# previously saved metrics summary when one is present.

def _recover_history(history_key, seed):
    path = METRIC_DIR / 'metrics_summary.json'
    try:
        prior = json.loads(path.read_text())
        for r in prior.get('runs', []):
            if int(r.get('seed', -1)) == int(seed) and r.get(history_key):
                return r[history_key]
    except Exception:
        pass
    return []

def load_or_train(name, seed, train, val, cfg, constrained, augment_fn=None,
                  history_key=None, train_seed=None):
    path = MODEL_DIR / f'{name}_seed{seed}.pt'
    node_dim = train[0].node_features.shape[1]
    edge_dim = train[0].edge_features.shape[1]
    if cfg.reuse_checkpoints and path.exists():
        model = EdgeMessagePassing(node_dim, edge_dim, cfg.hidden_dim, cfg.layers,
                                   cfg.num_regimes).to(DEVICE)
        try:
            model.load_state_dict(torch.load(path, map_location=DEVICE))
            model.eval()
            model.history = _recover_history(history_key, seed) if history_key else []
            model.best_epoch = None
            print(f'  loaded checkpoint {path.name} (training skipped)')
            return model
        except Exception as e:
            print(f'  checkpoint {path.name} not compatible ({e}); retraining')
    model = train_model(train, val, cfg, constrained=constrained,
                        seed=(train_seed if train_seed is not None else seed),
                        augment_fn=augment_fn)
    torch.save(model.state_dict(), path)
    print(f'  saved checkpoint {path.name}')
    return model

def run_single_seed(seed: int, cfg: Config) -> Dict[str, Any]:
    set_seed(seed)
    print('\n=== Running seed', seed, '===')
    train = make_dataset(cfg.train_n, seed=seed,     split='train', symmetry_orders=cfg.symmetry_orders)
    val   = make_dataset(cfg.val_n,   seed=seed+100, split='val',   symmetry_orders=cfg.symmetry_orders)
    test  = make_dataset(cfg.test_n,  seed=seed+200, split='test',  symmetry_orders=cfg.symmetry_orders)

    train_hf = make_dataset(max(24, cfg.train_n//2), seed=seed+300, split='train_hf',
                            symmetry_orders=cfg.symmetry_orders,
                            families=tuple(f for f in range(cfg.num_families) if f != cfg.heldout_family))
    test_hf  = make_dataset(max(8, cfg.test_n//2), seed=seed+400, split='test_hf',
                            symmetry_orders=cfg.symmetry_orders, families=(cfg.heldout_family,))
    test_hn  = make_dataset(max(8, cfg.test_n//2), seed=seed+600, split='test_hn',
                            symmetry_orders=(cfg.heldout_symmetry_order,))

    constrained_model = load_or_train('constrained', seed, train, val, cfg, True,
                                      history_key='history_constrained')
    unconstrained_model = load_or_train('unconstrained', seed, train, val, cfg, False,
                                        history_key='history_unconstrained', train_seed=seed+1)
    models = {'constrained': constrained_model, 'unconstrained': unconstrained_model}
    if cfg.train_augmented_variants:
        aug_c = make_missing_geometry_augmenter(cfg, seed)
        aug_u = make_missing_geometry_augmenter(cfg, seed + 1)
        models['constrained_aug'] = load_or_train('constrained_aug', seed, train, val, cfg, True,
                                                  augment_fn=aug_c,
                                                  history_key='history_constrained_aug')
        models['unconstrained_aug'] = load_or_train('unconstrained_aug', seed, train, val, cfg, False,
                                                    augment_fn=aug_u, train_seed=seed+1,
                                                    history_key='history_unconstrained_aug')

    pc, _       = calibrate_selection_policy(constrained_model, val, cfg, constrained=True)
    pu_orbit, _ = calibrate_selection_policy(unconstrained_model, val, cfg, constrained=False)
    pu_free, _  = calibrate_free_selection_policy(unconstrained_model, val, cfg)
    policies = {'constrained': pc, 'unconstrained_orbit_eval': pu_orbit, 'unconstrained_free': pu_free}
    if cfg.train_augmented_variants:
        pca, _       = calibrate_selection_policy(models['constrained_aug'], val, cfg, constrained=True)
        pua_orbit, _ = calibrate_selection_policy(models['unconstrained_aug'], val, cfg, constrained=False)
        pua_free, _  = calibrate_free_selection_policy(models['unconstrained_aug'], val, cfg)
        policies.update({'constrained_aug': pca, 'unconstrained_aug_orbit': pua_orbit,
                         'unconstrained_aug_free': pua_free})
    print('Policies:', {k: policy_to_label(v) for k, v in policies.items()})

    metrics = {
        'selection_policies': policies,
        'val_constrained':                evaluate_dataset(constrained_model, val, cfg, True,  policy=pc),
        'val_unconstrained_orbit_eval':   evaluate_dataset(unconstrained_model, val, cfg, False, policy=pu_orbit),
        'val_unconstrained_free':         evaluate_dataset(unconstrained_model, val, cfg, False, policy=pu_free),
        'test_constrained':               evaluate_dataset(constrained_model, test, cfg, True,  policy=pc),
        'test_unconstrained_orbit_eval':  evaluate_dataset(unconstrained_model, test, cfg, False, policy=pu_orbit),
        'test_unconstrained_free':        evaluate_dataset(unconstrained_model, test, cfg, False, policy=pu_free),
        'test_template':                  evaluate_baseline(test, cfg, deterministic_template_scores, threshold=0.5),
        'test_nearest_neighbor':          evaluate_baseline(test, cfg, lambda s: nearest_neighbor_scores(s, train), threshold=0.5),
    }
    if cfg.train_augmented_variants:
        metrics['test_constrained_aug'] = evaluate_dataset(models['constrained_aug'], test, cfg, True, policy=policies['constrained_aug'])
        metrics['test_unconstrained_aug_orbit_eval'] = evaluate_dataset(models['unconstrained_aug'], test, cfg, False, policy=policies['unconstrained_aug_orbit'])
        metrics['test_unconstrained_aug_free'] = evaluate_dataset(models['unconstrained_aug'], test, cfg, False, policy=policies['unconstrained_aug_free'])

    metrics['diagnostics'] = {
        'test_constrained': diagnose_run_health('test_constrained', metrics['test_constrained'], pc, cfg),
        'test_unconstrained_orbit_eval': diagnose_run_health('test_unconstrained_orbit_eval', metrics['test_unconstrained_orbit_eval'], pu_orbit, cfg),
        'test_unconstrained_free': diagnose_run_health('test_unconstrained_free', metrics['test_unconstrained_free'], pu_free, cfg),
        'constrained_vs_unconstrained_orbit_eval': compare_model_outputs_health(metrics['test_constrained'], metrics['test_unconstrained_orbit_eval']),
        'constrained_vs_unconstrained_free': compare_model_outputs_health(metrics['test_constrained'], metrics['test_unconstrained_free']),
    }
    metrics['theorem_invariants_constrained'] = verify_theorem_invariants_model(constrained_model, test, cfg, constrained=True, policy=pc)
    metrics['theorem_invariants_unconstrained_orbit_eval_diagnostic'] = verify_theorem_invariants_model(unconstrained_model, test, cfg, constrained=False, policy=pu_orbit)
    metrics['theorem_invariants_unconstrained_free_diagnostic'] = verify_theorem_invariants_model(unconstrained_model, test, cfg, constrained=False, policy=pu_free)
    if cfg.train_augmented_variants:
        metrics['theorem_invariants_constrained_aug'] = verify_theorem_invariants_model(models['constrained_aug'], test, cfg, constrained=True, policy=policies['constrained_aug'])
    print('construction-guarantee check (orbit-tied):', metrics['theorem_invariants_constrained']['status'])

    # ---- per-pattern fidelity records + label-free rotation audit ----
    f1_c, f1_p, f1_f = [], [], []
    rot_c, rot_f = [], []
    for s in test:
        sc, _ = predict_sample(constrained_model, s, constrained=True)
        su, _ = predict_sample(unconstrained_model, s, constrained=False)
        sel_c = select_with_policy(sc, s.orbit_ids, pc)
        sel_p = select_with_policy(su, s.orbit_ids, pu_orbit)
        sel_f = select_with_policy(su, s.orbit_ids, pu_free)
        f1_c.append(edge_metrics(s.edge_labels, sel_c)['f1'])
        f1_p.append(edge_metrics(s.edge_labels, sel_p)['f1'])
        f1_f.append(edge_metrics(s.edge_labels, sel_f)['f1'])
        rot_c.append(true_rotation_violations(s, sel_c))
        rot_f.append(true_rotation_violations(s, sel_f))
    metrics['rotation_audit'] = {
        'orbit_tied_true_violations_mean': float(np.mean(rot_c)),
        'orbit_tied_true_violations_max': float(np.max(rot_c)),
        'unstructured_free_true_violations_mean': float(np.mean(rot_f)),
        'num_samples': len(test),
    }
    print('label-free rotation audit (orbit-tied): mean violations =',
          metrics['rotation_audit']['orbit_tied_true_violations_mean'])

    if cfg.run_robustness:
        metrics['robustness'] = evaluate_robustness(
            constrained_model, unconstrained_model, test, cfg, pc, pu_orbit, pu_free, seed)
        rb = pd.DataFrame(metrics['robustness'])
        for _mode in cfg.robustness_modes:
            sm = rb[rb['mode'] == _mode]
            if len(sm):
                hi = sm[sm.severity == sm.severity.max()]
                print(f'  robustness[{_mode}] @ max severity  F1:',
                      {r.variant: round(r.f1, 3) for r in hi.itertuples()})
    if cfg.run_robustness and cfg.train_augmented_variants:
        metrics['robustness_training'] = evaluate_robustness_training(models, policies, test, cfg, seed)
        rt = pd.DataFrame(metrics['robustness_training'])
        hi = rt[rt.severity == rt.severity.max()]
        print('  training control @ max severity  F1:',
              {r.variant: round(r.f1, 3) for r in hi.itertuples()})

    metrics['heldout_family_constrained'] = evaluate_dataset(constrained_model, test_hf, cfg, True, policy=pc)
    metrics['heldout_family_unconstrained_orbit_eval'] = evaluate_dataset(unconstrained_model, test_hf, cfg, False, policy=pu_orbit)
    metrics['heldout_family_unconstrained_free'] = evaluate_dataset(unconstrained_model, test_hf, cfg, False, policy=pu_free)
    metrics['heldout_symmetry_constrained'] = evaluate_dataset(constrained_model, test_hn, cfg, True, policy=pc)
    metrics['heldout_symmetry_unconstrained_orbit_eval'] = evaluate_dataset(unconstrained_model, test_hn, cfg, False, policy=pu_orbit)
    metrics['heldout_symmetry_unconstrained_free'] = evaluate_dataset(unconstrained_model, test_hn, cfg, False, policy=pu_free)

    # ---- qualitative per-seed figures ----
    example = test[0]
    sc, ac = predict_sample(constrained_model, example, constrained=True)
    su, au = predict_sample(unconstrained_model, example, constrained=False)
    st = deterministic_template_scores(example)
    plot_prediction_grid(example, [
        (f'Orbit-tied\n{policy_to_label(pc)}', sc, ac, pc),
        (f'Projection (selection)\n{policy_to_label(pu_orbit)}', su, au, pu_orbit),
        (f'Unstructured (free)\n{policy_to_label(pu_free)}', su, au, pu_free),
        ('Procedural template', st, np.zeros_like(st), {'mode': 'threshold', 'threshold': 0.5}),
    ], cfg, path=FIG_DIR / f'synthesis_showcase_seed{seed}.png',
       title='From control geometry to completed pattern')
    export_prediction_svg(example, sc, ac, SVG_DIR / f'generated_constrained_seed{seed}.svg', cfg, policy=pc)

    plot_pretty_generation_gallery(constrained_model, cfg, pc, seed,
                                   FIG_DIR / f'raw_topology_gallery_seed{seed}.png')
    plot_regime_variety_gallery(constrained_model, cfg, pc, seed, FIG_DIR / f'regime_variety_seed{seed}.png')

    out = {'seed': seed, 'metrics': metrics,
           'history_constrained': constrained_model.history,
           'history_unconstrained': unconstrained_model.history}
    if cfg.train_augmented_variants:
        out['history_constrained_aug'] = models['constrained_aug'].history
        out['history_unconstrained_aug'] = models['unconstrained_aug'].history
    out['_models'] = models
    out['_constrained_model'] = constrained_model
    out['_unconstrained_model'] = unconstrained_model
    out['_val'] = val; out['_test'] = test
    out['_policies'] = policies
    out['_policy_constrained'] = pc; out['_pu_free'] = pu_free
    out['_paired'] = {'f1_orbit_tied': f1_c, 'f1_projection': f1_p, 'f1_free': f1_f}
    return out

# ---- run multi-seed main experiment ----
all_results = []
seeds_to_run = list(CFG.seeds) if CFG.run_multi_seed else [CFG.seed]
for sd in seeds_to_run:
    all_results.append(run_single_seed(sd, CFG))

# ---- aggregated quantitative figures ----
fig_main_f1(all_results, CFG, FIG_DIR / 'fig_main_f1.png')
fig_structural_validity(all_results, CFG, FIG_DIR / 'fig_structural_validity.png')
if all(len(r['history_constrained']) and len(r['history_unconstrained']) for r in all_results):
    fig_training_curves(all_results, CFG, FIG_DIR / 'fig_training_curves.png')
else:
    print('Training curves skipped: histories unavailable for loaded checkpoints '
          '(no prior metrics summary found).')

# operating curve from the first seed's constrained model
ref = all_results[0]
curve_df = topk_ratio_curve(ref['_constrained_model'], ref['_val'], CFG, constrained=True)
fig_selection_curve(curve_df, ref['_policy_constrained'].get('topk_ratio', 0.35), CFG, FIG_DIR / 'fig_selection_curve.png')
curve_df.to_csv(METRIC_DIR / 'selection_curve_seed{}.csv'.format(ref['seed']), index=False)

# robustness to imperfect control geometry
if CFG.run_robustness:
    fig_robustness(all_results, CFG, FIG_DIR / 'fig_robustness.png')
    fig_robustness_visual(ref, CFG, FIG_DIR / 'fig_robustness_visual.png')
    rb_all = pd.concat([pd.DataFrame(r['metrics']['robustness']) for r in all_results], ignore_index=True)
    rb_all.to_csv(METRIC_DIR / 'robustness.csv', index=False)
    rb_summary = (rb_all.groupby(['mode', 'severity', 'variant'])[['f1', 'orbit_violations', 'rot_sym_error']]
                  .agg(['mean', 'std']))
    print('\nRobustness summary (mean over seeds):')
    display(rb_summary)

# training-control study: clean-trained versus augmentation-trained variants
if CFG.run_robustness and CFG.train_augmented_variants:
    rt_all = pd.concat([pd.DataFrame(r['metrics']['robustness_training']) for r in all_results],
                       ignore_index=True)
    rt_all.to_csv(METRIC_DIR / 'robustness_training.csv', index=False)
    fig_robustness_training(rt_all, CFG, FIG_DIR / 'fig_robustness_training.png')
    rt_summary = (rt_all.groupby(['severity', 'variant'])[['f1', 'orbit_violations']]
                  .agg(['mean', 'std']))
    print('\nTraining-control summary (mean over seeds):')
    display(rt_summary)


## Statistical analysis of the fidelity comparison

In [ ]:
# ============================================================
# 18b. Statistical analysis of the fidelity comparison
# ============================================================
# Every decoding regime is evaluated on identical test patterns, so the fidelity
# comparison admits a paired per-pattern analysis. For each pair of methods the
# per-pattern F1 differences are summarised by their mean, a bootstrap 95%
# confidence interval over patterns, per-seed means, and a Wilcoxon signed-rank
# test. A pre-specified practical equivalence margin (CFG.equivalence_margin_f1)
# is used: the methods are reported as practically equivalent when the entire
# confidence interval lies inside the margin.
from scipy import stats as _scipy_stats

def paired_comparison(a, b, seeds_col, margin, n_boot=10000, rng_seed=20240):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    d = a - b
    rng = np.random.default_rng(rng_seed)
    boots = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    try:
        w = _scipy_stats.wilcoxon(d, zero_method='zsplit')
        w_stat, w_p = float(w.statistic), float(w.pvalue)
    except Exception:
        w_stat, w_p = float('nan'), float('nan')
    per_seed = {}
    for sd in sorted(set(seeds_col)):
        m = np.array([s == sd for s in seeds_col])
        per_seed[int(sd)] = float(d[m].mean())
    return {
        'n_patterns': int(len(d)),
        'mean_diff': float(d.mean()),
        'sd_diff': float(d.std(ddof=1)),
        'ci95_lo': float(lo), 'ci95_hi': float(hi),
        'wilcoxon_statistic': w_stat, 'wilcoxon_p': w_p,
        'per_seed_mean_diff': per_seed,
        'equivalence_margin': float(margin),
        'practically_equivalent': bool(-margin < lo and hi < margin),
    }

_f1c, _f1p, _f1f, _seedcol = [], [], [], []
for r in all_results:
    _f1c += list(r['_paired']['f1_orbit_tied'])
    _f1p += list(r['_paired']['f1_projection'])
    _f1f += list(r['_paired']['f1_free'])
    _seedcol += [r['seed']] * len(r['_paired']['f1_orbit_tied'])

paired_stats = {
    'orbit_tied_vs_unstructured_free': paired_comparison(_f1c, _f1f, _seedcol, CFG.equivalence_margin_f1),
    'orbit_tied_vs_projection':        paired_comparison(_f1c, _f1p, _seedcol, CFG.equivalence_margin_f1),
}
pd.DataFrame({'seed': _seedcol, 'f1_orbit_tied': _f1c,
              'f1_projection': _f1p, 'f1_free': _f1f}).to_csv(
    METRIC_DIR / 'paired_per_pattern_f1.csv', index=False)
save_json(paired_stats, METRIC_DIR / 'paired_stats.json')

for name, st in paired_stats.items():
    verdict = 'within' if st['practically_equivalent'] else 'NOT within'
    print(f"{name}: mean diff = {st['mean_diff']:+.4f}  "
          f"95% CI [{st['ci95_lo']:+.4f}, {st['ci95_hi']:+.4f}]  "
          f"Wilcoxon p = {st['wilcoxon_p']:.3g}  "
          f"-> {verdict} the +/-{st['equivalence_margin']:.2f} F1 equivalence margin "
          f"(n = {st['n_patterns']} patterns, per-seed means {st['per_seed_mean_diff']})")


## Per-N generalisation and data efficiency

In [ ]:
# ============================================================
# 19. Out-of-distribution per-N experiment + symmetry-pair figure
# ============================================================
def train_heldout_symmetry_pair(seed: int, cfg: Config):
    set_seed(seed)
    train_hn = make_dataset(max(24, cfg.train_n//2), seed=seed+500, split='train_hn',
                            symmetry_orders=cfg.train_symmetry_orders)
    val_hn   = make_dataset(max(8, cfg.val_n//2),   seed=seed+520, split='val_hn',
                            symmetry_orders=cfg.train_symmetry_orders)
    cm = load_or_train('constrained_hn', seed, train_hn, val_hn, cfg, True)
    um = load_or_train('unconstrained_hn', seed, train_hn, val_hn, cfg, False, train_seed=seed+1)
    pc, _ = calibrate_selection_policy(cm, val_hn, cfg, constrained=True)
    pu, _ = calibrate_selection_policy(um, val_hn, cfg, constrained=False)
    return cm, um, pc, pu

per_n_results = []
first_pair = None
for i, sd in enumerate(seeds_to_run):
    cm, um, pc, pu = train_heldout_symmetry_pair(sd, CFG)
    per_n_results.append({
        'seed': sd,
        'constrained':   evaluate_per_n(cm, CFG, True,  pc, sd),
        'unconstrained': evaluate_per_n(um, CFG, False, pu, sd),
    })
    print(f'per-N seed {sd}: constrained N={CFG.heldout_symmetry_order} '
          f"F1={per_n_results[-1]['constrained'][CFG.heldout_symmetry_order]['f1']:.3f}  "
          f"unconstrained F1={per_n_results[-1]['unconstrained'][CFG.heldout_symmetry_order]['f1']:.3f}")
    if first_pair is None:
        first_pair = (cm, um, pc, pu, sd)

fig_per_n(per_n_results, CFG, FIG_DIR / 'fig_per_n.png')
fig_symmetry_pair(first_pair, CFG, FIG_DIR / 'fig_symmetry_pair.png')

# per-N table
pern_rows = []
for N in CFG.symmetry_orders:
    c = np.array([r['constrained'][int(N)]['f1'] for r in per_n_results])
    u = np.array([r['unconstrained'][int(N)]['f1'] for r in per_n_results])
    pern_rows.append({'N': int(N), 'held_out': bool(N == CFG.heldout_symmetry_order),
                      'constrained_f1': f'{c.mean():.4f} ± {c.std():.4f}',
                      'unconstrained_f1': f'{u.mean():.4f} ± {u.std():.4f}'})
df_pern = pd.DataFrame(pern_rows); display(df_pern)
df_pern.to_csv(METRIC_DIR / 'per_n_table.csv', index=False)

# ============================================================
# 20. Data-efficiency ablation
# ============================================================
if CFG.run_data_efficiency:
    de_cache = METRIC_DIR / 'data_efficiency.csv'
    if CFG.reuse_cached_ablations and de_cache.exists():
        de_df = pd.read_csv(de_cache)
        print('Data-efficiency ablation loaded from cache:', de_cache,
              '(delete the file to recompute).')
    else:
        de_df = run_data_efficiency(CFG)
        de_df.to_csv(de_cache, index=False)
    fig_data_efficiency(de_df, CFG, FIG_DIR / 'fig_data_efficiency.png')
    display(de_df)
else:
    de_df = None


## Ornament presentation rendering (styling only)

In [ ]:
# ============================================================
# 21b. Ornament presentation options (styling only; geometry unaltered)
# ============================================================
CFG.ornament_bg          = '#f5eedb'   # parchment
CFG.ornament_band        = '#1d3557'   # deep indigo  (frame tier)
CFG.ornament_accent      = '#9a7426'   # deep gold    (star tier, drawn last)
CFG.ornament_minor       = '#b3bac6'   # pale slate   (minor tier, demoted)
CFG.ornament_band_width  = 3.0
CFG.ornament_casing_ratio= 1.75
CFG.ornament_best_of_n   = 14          # best-of-N curation for qualitative panels only
CFG.ornament_density_lo  = 0.14
CFG.ornament_density_hi  = 0.30

# ============================================================
# 13. Ornament presentation rendering (strapwork styling)
# ============================================================
# PRESENTATION ONLY: restyles the model's selected, refined polylines as woven
# strapwork bands (a wide background-colour casing stroke under a coloured band
# stroke, drawn per edge in a fixed shuffled order so crossings read as woven).
# No geometry is added, removed, or moved; the woven look is purely draw order.
# Best-of-N curation is for qualitative panels only, never quantitative tables.

ORNAMENT_TIER_OF_TYPE = {
    # skip edges are the {N/k} star-polygon chords of classical construction -> star tier
    EDGE_TYPES['skip']: 'star', EDGE_TYPES['star_cross']: 'star',
    EDGE_TYPES['ring']: 'frame', EDGE_TYPES['radial']: 'frame',
    EDGE_TYPES['boundary']: 'frame', EDGE_TYPES['border']: 'frame',
    EDGE_TYPES['diag']: 'minor', EDGE_TYPES['secondary']: 'minor',
}
# Visual grammar (styling only; declared in the README): minor connective lines are
# thin, pale, and casing-free; frame straps are indigo with woven casing; star bands
# are widest, gold, drawn last so they weave over everything else.
ORNAMENT_TIER_STYLE = {
    'minor': dict(width_mult=0.40, colour_key='minor',  casing=False, order=0),
    'frame': dict(width_mult=0.90, colour_key='band',   casing=True,  order=1),
    'star':  dict(width_mult=0.95, colour_key='accent', casing=True,  order=2),
}

def _ornament_style(cfg, band_w=None):
    bw = float(band_w or cfg.ornament_band_width)
    return {'bg': cfg.ornament_bg, 'band': cfg.ornament_band, 'accent': cfg.ornament_accent,
            'minor': cfg.ornament_minor, 'band_w': bw,
            'casing_ratio': float(cfg.ornament_casing_ratio)}

def refined_polylines_with_types(sample, selected, alpha, cfg):
    polys = refined_edge_polylines(sample.points, sample.edges, selected, alpha,
                                   sample.edge_types, cfg.alpha_max)
    types = [int(t) for t, s in zip(sample.edge_types, selected) if s]
    return polys, types

def _tier_sequence(polylines, types):
    """Group polyline indices by tier, each group deterministically shuffled."""
    groups = {'minor': [], 'frame': [], 'star': []}
    for i, t in enumerate(types):
        groups[ORNAMENT_TIER_OF_TYPE.get(int(t), 'frame')].append(i)
    rng = np.random.default_rng(7919 * (len(polylines) + 1))
    seq = []
    for tier in sorted(ORNAMENT_TIER_STYLE, key=lambda k: ORNAMENT_TIER_STYLE[k]['order']):
        idx = groups[tier]
        rng.shuffle(idx)
        seq.extend((i, tier) for i in idx)
    return seq

def draw_strapwork(ax, polylines, types, cfg, limit=None, title=None, band_w=None):
    style = _ornament_style(cfg, band_w)
    limit = float(limit or cfg.gallery_axis_limit)
    ax.set_facecolor(style['bg'])
    ax.set_aspect('equal'); ax.set_xlim(-limit, limit); ax.set_ylim(-limit, limit)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    for i, tier in _tier_sequence(polylines, types):
        ts = ORNAMENT_TIER_STYLE[tier]
        pl = np.asarray(polylines[i])
        w = style['band_w'] * ts['width_mult']
        col = style[ts['colour_key']]
        if ts['casing']:
            ax.plot(pl[:, 0], pl[:, 1], color=style['bg'], lw=w * style['casing_ratio'],
                    solid_capstyle='butt', solid_joinstyle='round', zorder=2)
        ax.plot(pl[:, 0], pl[:, 1], color=col, lw=w,
                solid_capstyle='round', solid_joinstyle='round', zorder=2)
    if title:
        ax.set_title(title, fontsize=10.5, pad=8, color='#3a3326')

def _save_ornament(fig, path):
    path = Path(path).with_suffix('.png')
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=320, bbox_inches='tight', facecolor=fig.get_facecolor())
    print('Saved figure:', path)
    return path

def ornament_quality(sample, scores, alpha, policy, cfg):
    """Curation score for qualitative panels only: prefers a sparser density window,
    rotational self-consistency, and edge-type variety. Uses no target labels."""
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    density = float(selected.mean())
    lo, hi = cfg.ornament_density_lo, cfg.ornament_density_hi
    mid = 0.5 * (lo + hi)
    density_score = max(0.0, 1.0 - abs(density - mid) / max(1e-6, 1.6 * (hi - lo)))
    polys = refined_edge_polylines(sample.points, sample.edges, selected, alpha,
                                   sample.edge_types, cfg.alpha_max)
    pts = [p[::max(1, len(p)//4)] for p in polys[:300] if len(p)]
    sym_score = 0.0
    if pts:
        P = np.concatenate(pts, axis=0)
        if len(P) > 500:
            P = P[:: int(np.ceil(len(P) / 500))]
        sym = rotational_self_chamfer(P, sample.N)
        sym_score = max(0.0, 1.0 - sym / 0.05)
    sel_types = [int(t) for t, s in zip(sample.edge_types, selected) if s]
    diversity = len(set(sel_types)) / float(NUM_EDGE_TYPES)
    lens = [float(np.linalg.norm(sample.points[int(b)] - sample.points[int(a)]))
            for (a, b), s in zip(sample.edges, selected) if s]
    outer = float(sample.scaffold_info.get('outer_radius', 1.0))
    length_score = min(1.0, (np.mean(lens) / (0.42 * outer))) if lens else 0.0
    # visual-grammar composition: reward dominant star/frame structure, penalise
    # panels dominated by minor connective lines (crossing-diagonal clutter)
    n_sel = max(1, len(sel_types))
    frac_star = sum(1 for t in sel_types if ORNAMENT_TIER_OF_TYPE.get(t) == 'star') / n_sel
    frac_minor = sum(1 for t in sel_types if ORNAMENT_TIER_OF_TYPE.get(t) == 'minor') / n_sel
    star_balance = max(0.0, 1.0 - abs(frac_star - 0.33) / 0.33)   # gold present but not engulfing
    minor_calm = max(0.0, 1.0 - max(0.0, frac_minor - 0.25) / 0.50)
    hierarchy = 0.6 * star_balance + 0.4 * minor_calm
    return (0.28 * density_score + 0.24 * sym_score + 0.15 * length_score
            + 0.23 * hierarchy + 0.10 * diversity)

def pick_ornament_sample(model, cfg, policy, N, scaffold_type, family_id, regime_id, rng, attempts=None):
    attempts = int(attempts or cfg.ornament_best_of_n)
    best = None
    for j in range(max(1, attempts)):
        sample = make_sample(f'orn_{N}_{family_id}_{regime_id}_{j}', N, scaffold_type,
                             family_id, regime_id, rng)
        scores, alpha = predict_sample(model, sample, constrained=True)
        q = ornament_quality(sample, scores, alpha, policy, cfg)
        if best is None or q > best[-1]:
            best = (sample, scores, alpha, q)
    return best

def export_strapwork_svg(polylines, types, cfg, path, limit=None, size=1000):
    style = _ornament_style(cfg)
    limit = float(limit or cfg.gallery_axis_limit)
    s = size / (2.0 * limit)
    base_px = size * 0.0072
    def tx(p): return ((p[0] + limit) * s, (limit - p[1]) * s)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {size} {size}">',
             f'<rect width="100%" height="100%" fill="{style["bg"]}"/>']
    case_common = 'fill="none" stroke-linecap="butt" stroke-linejoin="round"'
    band_common = 'fill="none" stroke-linecap="round" stroke-linejoin="round"'
    for i, tier in _tier_sequence(polylines, types):
        ts = ORNAMENT_TIER_STYLE[tier]
        pts = " ".join(f"{x:.2f},{y:.2f}" for x, y in (tx(p) for p in np.asarray(polylines[i])))
        w = base_px * ts['width_mult']
        col = style[ts['colour_key']]
        if ts['casing']:
            parts.append(f'<polyline points="{pts}" {case_common} stroke="{style["bg"]}" '
                         f'stroke-width="{w * style["casing_ratio"]:.2f}"/>')
        parts.append(f'<polyline points="{pts}" {band_common} stroke="{col}" stroke-width="{w:.2f}"/>')
    parts.append('</svg>')
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(parts), encoding='utf-8')
    print('Saved SVG:', path)

def plot_ornament_gallery(model, cfg, policy, seed, path):
    rng = np.random.default_rng(seed + 7373)
    specs = [(6, 'anchored', 0, 0), (8, 'anchored', 1, 2), (10, 'anchored', 3, 1),
             (12, 'anchored', 4, 3), (8, 'minimal', 2, 3), (12, 'minimal', 0, 1)]
    fig, axes = plt.subplots(2, 3, figsize=(12.6, 8.8), facecolor=cfg.ornament_bg,
                             constrained_layout=False)
    axes = axes.ravel()
    for ax, (N, scaffold_type, family_id, regime_id) in zip(axes, specs):
        sample, scores, alpha, q = pick_ornament_sample(model, cfg, policy, N, scaffold_type,
                                                        family_id, regime_id, rng)
        selected = select_with_policy(scores, sample.orbit_ids, policy)
        polys, types = refined_polylines_with_types(sample, selected, alpha, cfg)
        draw_strapwork(ax, polys, types, cfg, title=f'$N={N}$ · {scaffold_type}')
        if getattr(cfg, 'save_gallery_svgs', True):
            export_strapwork_svg(polys, types, cfg,
                                 SVG_DIR / f'ornament_N{N}_{scaffold_type}_f{family_id}_seed{seed}.svg')
    fig.tight_layout(pad=1.2)
    return _save_ornament(fig, path)

def plot_ornament_hero(model, cfg, policy, seed, path, svg_path=None):
    rng = np.random.default_rng(seed + 8484)
    combos = [(12, 'anchored', 4, 3), (12, 'anchored', 1, 2), (12, 'anchored', 0, 0),
              (10, 'anchored', 3, 1), (10, 'anchored', 2, 0),
              (8, 'anchored', 1, 2), (8, 'anchored', 4, 1)]
    best, best_combo = None, None
    for N, st, fam, reg in combos:
        cand = pick_ornament_sample(model, cfg, policy, N, st, fam, reg, rng)
        if best is None or cand[-1] > best[-1]:
            best, best_combo = cand, (N, st, fam, reg)
    sample, scores, alpha, q = best
    selected = select_with_policy(scores, sample.orbit_ids, policy)
    polys, types = refined_polylines_with_types(sample, selected, alpha, cfg)
    fig, ax = plt.subplots(figsize=(8.8, 8.8), facecolor=cfg.ornament_bg, constrained_layout=False)
    draw_strapwork(ax, polys, types, cfg, limit=cfg.gallery_axis_limit * 0.98,
                   band_w=cfg.ornament_band_width * 1.55)
    fig.tight_layout(pad=0.4)
    if svg_path is not None:
        export_strapwork_svg(polys, types, cfg, svg_path)
    out = _save_ornament(fig, path)
    print(f'  hero: N={best_combo[0]}, family={best_combo[2]}, regime={best_combo[3]}, curation score={q:.3f}')
    return out


## Ornament galleries and hero panels

In [ ]:
# ============================================================
# 22. Ornament presentation figures (styling only; geometry unaltered)
# ============================================================
for r in all_results:
    sd = r['seed']
    cm = r['_constrained_model']
    pc = r['_policies']['constrained']
    plot_ornament_gallery(cm, CFG, pc, sd, FIG_DIR / f'ornament_gallery_seed{sd}.png')
    plot_ornament_hero(cm, CFG, pc, sd, FIG_DIR / f'ornament_hero_seed{sd}.png',
                       svg_path=SVG_DIR / f'ornament_hero_seed{sd}.svg')
print('Ornament galleries and hero panels written to', FIG_DIR)


## Results tables and summary

In [ ]:
# ============================================================
# 23. Main results table + save everything
# ============================================================
def flatten_metric_runs(results, key):
    vals = {}
    for m in results[0]['metrics'][key].keys():
        arr = np.array([r['metrics'][key][m] for r in results], dtype=float)
        vals[m] = f'{arr.mean():.4f} ± {arr.std():.4f}'
    return vals

main_keys = ['test_constrained', 'test_unconstrained_orbit_eval', 'test_unconstrained_free',
             'test_constrained_aug', 'test_unconstrained_aug_orbit_eval', 'test_unconstrained_aug_free',
             'test_template', 'test_nearest_neighbor',
             'heldout_family_constrained', 'heldout_family_unconstrained_orbit_eval', 'heldout_family_unconstrained_free',
             'heldout_symmetry_constrained', 'heldout_symmetry_unconstrained_orbit_eval', 'heldout_symmetry_unconstrained_free']
rows = []
for key in main_keys:
    if key in all_results[0]['metrics']:
        flat = flatten_metric_runs(all_results, key)
        rows.append({'setting': key, 'f1': flat.get('f1'), 'precision': flat.get('precision'),
                     'recall': flat.get('recall'), 'p@k': flat.get('precision_at_k'),
                     'chamfer': flat.get('chamfer'), 'hausdorff': flat.get('hausdorff'),
                     'rot_self_cd': flat.get('rot_self_chamfer'), 'density_err': flat.get('edge_density_error')})
df_results = pd.DataFrame(rows); display(df_results)
df_results.to_csv(METRIC_DIR / 'main_results_table.csv', index=False)

# structural-validity summary table (label-based invariants + label-free audit)
sv_rows = []
for r in all_results:
    m = r['metrics']
    pairs = [('orbit_tied', 'theorem_invariants_constrained'),
             ('projection_selection', 'theorem_invariants_unconstrained_orbit_eval_diagnostic'),
             ('unstructured_free', 'theorem_invariants_unconstrained_free_diagnostic')]
    if 'theorem_invariants_constrained_aug' in m:
        pairs.append(('orbit_tied_aug', 'theorem_invariants_constrained_aug'))
    for tag, key in pairs:
        t = m[key]
        sv_rows.append({'seed': r['seed'], 'model': tag,
                        'orbit_violations': t.get('selected_orbit_violation_count', 0),
                        'anchor_drift': t.get('max_anchor_coordinate_drift', 0.0),
                        'alpha_bound_excess': t.get('max_alpha_bound_excess', 0.0),
                        'num_samples': t.get('num_samples', 0)})
    sv_rows.append({'seed': r['seed'], 'model': 'orbit_tied_rotation_audit',
                    'orbit_violations': m['rotation_audit']['orbit_tied_true_violations_mean'],
                    'anchor_drift': 0.0, 'alpha_bound_excess': 0.0,
                    'num_samples': m['rotation_audit']['num_samples']})
df_sv = pd.DataFrame(sv_rows); display(df_sv)
df_sv.to_csv(METRIC_DIR / 'structural_validity.csv', index=False)

# strip in-memory handles before JSON serialization
clean_runs = []
for r in all_results:
    clean_runs.append({k: v for k, v in r.items() if not k.startswith('_')})

summary = {'config': {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.__dict__.items()},
           'runs': clean_runs,
           'per_n': per_n_results,
           'data_efficiency': (de_df.to_dict(orient='records') if de_df is not None else None),
           'paired_stats': paired_stats}
save_json(summary, METRIC_DIR / 'metrics_summary.json')
print('Saved metrics summary:', METRIC_DIR / 'metrics_summary.json')


## Release notes

In [ ]:
# ============================================================
# 24. Release notes written next to the results
# ============================================================
README = """Islamic Geometric Patterns through Symmetry-Structured Neural Completion
release notebook (v10)

This notebook trains, evaluates, and renders the complete system reported in
the paper. It is incremental: model checkpoints found in Results/models are
loaded and their training is skipped, so a completed run is extended rather
than repeated. Delete a checkpoint to retrain that model. Cached ablation
outputs (data_efficiency.csv) are reused in the same way.

Orbit identifiers are the exact geometric orbit partition, computed by
union-find under the C_N rotation map on vertex coordinates. The symmetry
guarantee is verified by a label-free audit that rotates every selected edge
by exactly 2*pi/N and checks that its image is selected, independently of the
identifiers the system itself uses.

Model variants
  constrained        : orbit-tied predictor (guarantee by construction)
  unconstrained      : ordinary predictor; evaluated free, with orbit-level
                       selection, and with full orbit projection
  constrained_aug    : orbit-tied predictor trained with missing-control-
                       geometry augmentation
  unconstrained_aug  : ordinary predictor trained with the same augmentation
The augmented variants separate the contribution of symmetry structure from
the contribution of the training distribution under corrupted input.

Key outputs (Results/)
  metrics/main_results_table.csv     main fidelity/calibration table
  metrics/structural_validity.csv    invariants incl. label-free rotation audit
  metrics/robustness.csv             corruption study (clean-trained models)
  metrics/robustness_training.csv    training-control study (augmented models)
  metrics/paired_per_pattern_f1.csv  per-pattern paired fidelity records
  metrics/paired_stats.json          paired differences, bootstrap CIs,
                                     Wilcoxon tests, equivalence verdicts
  metrics/metrics_summary.json       complete run record
  figures/                           all paper figures (quantitative +
                                     ornament galleries and hero panels)
  svg/                               scalable vector exports
  models/                            trained checkpoints (reused on re-run)

Presentation rendering is styling only. The strapwork figures restyle the
model's selected, refined polylines as woven bands with a declared visual
hierarchy mapped from the model's own edge types. No geometry is added,
removed, or moved, and the unstyled topological completions are saved beside
every styled panel (raw_topology_gallery_*).
"""
(RESULTS_ROOT / 'README_release.md').write_text(README, encoding='utf-8')
print(README)
